In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path(
    "../data/processed/cleaned_grading_dataset.xlsx"
)

df = pd.read_excel(DATA_PATH)

print("Dataset shape:", df.shape)
print("Columns:", len(df.columns))

print(df.head())

Dataset shape: (1055, 116)
Columns: 116
                      Tissue_ID    Eye                Image_ID Disease_Label  \
0  12346 R (year/month unknown)  Right                     NaN        Normal   
1           NEB-2020-09-12654 R  Right  NEB-2020-09-12654 R_SP        Normal   
2           NEB-2020-09-12655 L   Left  NEB-2020-09-12655 L_SP         Folds   
3           NEB-2020-09-12656 R  Right  NEB-2020-09-12656 R_SP        Normal   
4           NEB-2020-09-12657 L   Left  NEB-2020-09-12657 L_SP         Folds   

  Disease_Severity  ECD_CD_cells_per_mm2    CV  HEX_percent     SD  \
0              NaN                   NaN   NaN          NaN    NaN   
1              NaN                2132.0  33.0         58.0  153.0   
2         Moderate                2212.0  33.0         47.0  147.0   
3              NaN                2538.0  35.0         53.0  137.0   
4             Mild                2584.0  29.0         69.0  112.0   

   Total_Cell_Count  ...  Epi_Has_Exposure Epi_Exposure_Se

In [3]:
TARGET = "Tissue_Grade_Clean"

assert TARGET in df.columns, f"Missing target column: {TARGET}"

print("Target value counts, including missing values:")
display(df[TARGET].value_counts(dropna=False))

print("\nTarget percentages among labelled records:")
display(
    (df[TARGET].value_counts(normalize=True) * 100).round(2)
)

print("\nMissing target values:", df[TARGET].isna().sum())
print("Labelled records:", df[TARGET].notna().sum())

Target value counts, including missing values:


Tissue_Grade_Clean
A+     545
A      319
B      172
NaN     19
Name: count, dtype: int64


Target percentages among labelled records:


Tissue_Grade_Clean
A+    52.61
A     30.79
B     16.60
Name: proportion, dtype: float64


Missing target values: 19
Labelled records: 1036


In [4]:
VALID_GRADES = ["A+", "A", "B"]

labelled_df = df[df[TARGET].isin(VALID_GRADES)].copy()

print("Original rows:", len(df))
print("Rows with valid specialist grades:", len(labelled_df))
print("Excluded rows with missing or unexpected grades:",
      len(df) - len(labelled_df))

display(labelled_df[TARGET].value_counts().rename("Count").to_frame())

Original rows: 1055
Rows with valid specialist grades: 1036
Excluded rows with missing or unexpected grades: 19


,Count
Tissue_Grade_Clean,
A+,545
A,319
B,172


In [5]:
identifier_candidates = [
    "Tissue_ID",
    "Eye",
    "Image_ID",
    "Donor_ID",
    "Donor_ID_Clean",
]

identifier_summary = []

for col in identifier_candidates:
    if col in labelled_df.columns:
        identifier_summary.append({
            "Column": col,
            "Non-null": labelled_df[col].notna().sum(),
            "Unique values": labelled_df[col].nunique(dropna=True),
            "Missing": labelled_df[col].isna().sum(),
        })

display(pd.DataFrame(identifier_summary))

,Column,Non-null,Unique values,Missing
0,Tissue_ID,1036,1036,0
1,Eye,1036,2,0
2,Image_ID,999,999,37


In [6]:
potential_leakage_columns = [
    "Overall_Donor_Quality",
    "Transplant_Suitability",
    "Suitability_Reason",
    "Tissue_Outcome",
    "Recipient_Age",
    "Recipient_Eye",
    "Recipient_Indication",
    "Request_Indication",
    "Primary_Procedure",
    "Secondary_Procedures",
    "Surgery_Date",
    "Requested_Grade",
    "Planned_Procedure",
    "Grade_Request_Met",
    "Review_Notes",
    "Additional_Findings",
]

leakage_columns_present = [
    col for col in potential_leakage_columns
    if col in labelled_df.columns
]

print("Potential leakage/context columns found:")
for col in leakage_columns_present:
    print("-", col)

print("\nColumns not found in this dataset:")
for col in potential_leakage_columns:
    if col not in labelled_df.columns:
        print("-", col)

Potential leakage/context columns found:
- Overall_Donor_Quality
- Transplant_Suitability
- Suitability_Reason
- Tissue_Outcome
- Recipient_Age
- Recipient_Eye
- Recipient_Indication
- Request_Indication
- Primary_Procedure
- Secondary_Procedures
- Surgery_Date
- Requested_Grade
- Planned_Procedure
- Grade_Request_Met
- Review_Notes
- Additional_Findings

Columns not found in this dataset:


In [7]:
numeric_candidates = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
    "Donor_Age",
]

clinical_candidates = [
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Has_Scar",
    "Stroma_Arcus_Severity",
    "Stroma_Scar_Type",
    "Descemet_Has_Folds",
    "Descemet_Fold_Severity",
    "Descemet_Fold_Relative_Number",
    "Endo_Has_Stress_Lines",
    "Endo_Stress_Line_Location",
    "Endo_Stress_Line_Amount",
    "Endo_Has_Cell_Dropout",
    "Endo_Cell_Dropout_Severity",
    "Endo_Has_RBC",
    "Endo_RBC_Severity",
    "Epi_Has_Exposure",
    "Epi_Exposure_Severity",
    "Epi_Exposure_Location",
    "Epi_Has_Defect",
    "Epi_Has_Haze",
    "Epi_Haze_Severity",
    "Epi_Has_Arcus",
]

numeric_present = [
    col for col in numeric_candidates
    if col in labelled_df.columns
]

clinical_present = [
    col for col in clinical_candidates
    if col in labelled_df.columns
]

print("Numeric candidates:", len(numeric_present))
print(numeric_present)

print("\nClinical candidates:", len(clinical_present))
print(clinical_present)

print("\nMissing candidate columns:")
print("Numeric:", sorted(set(numeric_candidates) - set(numeric_present)))
print("Clinical:", sorted(set(clinical_candidates) - set(clinical_present)))

Numeric candidates: 7
['ECD_CD_cells_per_mm2', 'CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2', 'Donor_Age']

Clinical candidates: 24
['Stroma_Is_Clear', 'Stroma_Is_Compact', 'Stroma_Has_Arcus', 'Stroma_Has_Edema', 'Stroma_Has_Scar', 'Stroma_Arcus_Severity', 'Stroma_Scar_Type', 'Descemet_Has_Folds', 'Descemet_Fold_Severity', 'Descemet_Fold_Relative_Number', 'Endo_Has_Stress_Lines', 'Endo_Stress_Line_Location', 'Endo_Stress_Line_Amount', 'Endo_Has_Cell_Dropout', 'Endo_Cell_Dropout_Severity', 'Endo_Has_RBC', 'Endo_RBC_Severity', 'Epi_Has_Exposure', 'Epi_Exposure_Severity', 'Epi_Exposure_Location', 'Epi_Has_Defect', 'Epi_Has_Haze', 'Epi_Haze_Severity', 'Epi_Has_Arcus']

Missing candidate columns:
Numeric: []
Clinical: []


In [8]:
identifier_like = [
    col for col in df.columns
    if any(term in col.lower() for term in [
        "donor", "tissue_id", "image_id", "serial",
        "accession", "specimen", "record_id"
    ])
]

print("Identifier-like columns:")
for col in identifier_like:
    print(f"\n{col}")
    print("  dtype:", df[col].dtype)
    print("  non-null:", df[col].notna().sum())
    print("  unique:", df[col].nunique(dropna=True))
    print("  examples:", df[col].dropna().astype(str).head(5).tolist())

Identifier-like columns:

Tissue_ID
  dtype: str
  non-null: 1055
  unique: 1055
  examples: ['12346 R (year/month unknown)', 'NEB-2020-09-12654 R', 'NEB-2020-09-12655 L', 'NEB-2020-09-12656 R', 'NEB-2020-09-12657 L']

Image_ID
  dtype: str
  non-null: 1013
  unique: 1013
  examples: ['NEB-2020-09-12654 R_SP', 'NEB-2020-09-12655 L_SP', 'NEB-2020-09-12656 R_SP', 'NEB-2020-09-12657 L_SP', 'NEB-2020-09-12658 R_SP']

Overall_Donor_Quality
  dtype: str
  non-null: 275
  unique: 24
  examples: ['Excellent', 'Excellent', 'Excellent', 'Excellent', 'Excellent']

Donor_Age
  dtype: float64
  non-null: 1050
  unique: 57
  examples: ['57.0', '57.0', '51.0', '51.0', '55.0']

Tissue_Serial
  dtype: int64
  non-null: 1055
  unique: 1054
  examples: ['12346', '12654', '12655', '12656', '12657']

Donor_Sex
  dtype: str
  non-null: 566
  unique: 22
  examples: ['Male', 'Male', 'Female', 'Female', 'Female']

Donor_Graft_Size_mm
  dtype: object
  non-null: 234
  unique: 29
  examples: ['8.25', '8', '8.5',

In [9]:
id_audit = labelled_df[["Tissue_ID"]].copy()

id_audit["Tissue_ID"] = id_audit["Tissue_ID"].astype("string").str.strip()

print("Unique tissue IDs:", id_audit["Tissue_ID"].nunique())
print("Duplicate tissue IDs:", id_audit["Tissue_ID"].duplicated().sum())

display(
    id_audit["Tissue_ID"]
    .str.extract(r"^(.*?)(?:\s+[RL])?$", expand=False)
    .value_counts()
    .head(20)
)

Unique tissue IDs: 1036
Duplicate tissue IDs: 0


Tissue_ID
NEB-2020-09-12654    1
NEB-2020-09-12655    1
NEB-2020-09-12656    1
NEB-2020-09-12657    1
NEB-2020-09-12658    1
NEB-2020-09-12659    1
NEB-2020-09-12660    1
NEB-2020-09-12661    1
NEB-2020-09-12662    1
NEB-2020-09-12663    1
NEB-2020-09-12664    1
NEB-2020-09-12665    1
NEB-2020-09-12666    1
NEB-2020-09-12667    1
NEB-2020-09-12668    1
NEB-2020-09-12669    1
NEB-2020-09-12670    1
NEB-2020-09-12671    1
NEB-2020-09-12672    1
NEB-2020-09-12673    1
Name: count, dtype: Int64

In [10]:
for col in [
    "Overall_Donor_Quality",
    "Transplant_Suitability",
    "Requested_Grade",
    "Grade_Request_Met",
    "Suitability_Reason",
    "Tissue_Outcome",
    "Review_Notes",
    "Additional_Findings",
]:
    if col in labelled_df.columns:
        print(f"\n--- {col} ---")
        display(labelled_df[col].value_counts(dropna=False).head(15))


--- Overall_Donor_Quality ---


Overall_Donor_Quality
NaN                                                                762
Excellent                                                          136
Very good                                                           79
Good                                                                31
Very good (as read)                                                  4
Excellent (as read)                                                  2
Good (as read)                                                       2
Good (Very good struck)                                              2
Excellent / very good (mark spans both - verify)                     2
Very good (Good also marked - verify)                                1
Very good (stroke runs across Excellent and Very good - verify)      1
Very good (as read - marks also near Excellent/Good)                 1
Excellent / Very good (stroke crosses both - verify)                 1
Good (as read - tick trails from Very good)            


--- Transplant_Suitability ---


Transplant_Suitability
NaN                                                                                                          582
Suitable (released for surgery - printed release form)                                                       197
Suitable                                                                                                     175
Not stated (no release form in PDF); donor sheet grade A+                                                      9
Not ticked on Donor Form B                                                                                     9
Released (printed release form); Donor Form B suitability unticked                                             7
Suitable (Donor Form B; released for surgery - printed release form)                                           7
Suitable (released for surgery - printed release form); Donor Form B stroke crosses suitable/not suitable      5
Not stated (no release form in PDF); donor sheet grade A                 


--- Requested_Grade ---


...

Tissue_Outcome
NaN                                                                                      986
Exported - cornea sent to Dr Y.Y. Choong, Malaysia, 23/09/2025 (handwritten note)          3
Allocated to Dr Pattaramon, Rutnin Medical, Thailand (29/09/2020)                          2
Cornea expired on 03.01.2025 (handwritten)                                                 2
Exported - cornea sent to KPJ Eye Specialist, Malaysia, 26/12/2024 (handwritten note)      2
Cornea expired 16/10/2025 (handwritten note)                                               2
Cornea expired on 23.10.2025 (handwritten)                                                 2
Cornea expired on 28.10.2025 (handwritten)                                                 2
Cornea expired on 31/10/2025 (handwritten)                                                 2
Cornea expired 11/11/2025 (handwritten note)                                               2
Discarded - Hep C positive (handwritten note)          


--- Review_Notes ---


Review_Notes
NaN                                                                                                                                             749
Folds call uncertain                                                                                                                             43
Guttata call uncertain                                                                                                                           16
CD 2538 within 50 of the 2,500 A/A+ cut-off                                                                                                      10
Guttata call uncertain; Guttata label (reference: ~50% A at CD ≥ 2,500)                                                                           7
Guttata label (reference: ~50% A at CD ≥ 2,500)                                                                                                   7
CD in 2200–2300 borderline zone; CD 2268 within 50 of the 2,250 B/A cut-off                        


--- Additional_Findings ---


Additional_Findings
NaN                                                                                                                                                         464
Rim 3-4; cornea diameter 11 mm; corneoscleral 18 mm                                                                                                          60
Rim 3-3.5; cornea diameter 11 mm; corneoscleral 18 mm                                                                                                        16
Rim 2.5-4; cornea diameter 11 mm; corneoscleral 18 mm                                                                                                        10
Rim 3-5; cornea diameter 11 mm; corneoscleral 18 mm                                                                                                          10
Rim 2-3; cornea diameter 11 mm; corneoscleral 18 mm                                                                                                           9
Rim 2.5-3.5; cornea 

In [11]:
serial_col = "Tissue_Serial"

print("Rows:", len(labelled_df))
print("Unique serials:", labelled_df[serial_col].nunique())
print("Duplicate serial rows:", labelled_df[serial_col].duplicated(keep=False).sum())

duplicate_serials = labelled_df[
    labelled_df[serial_col].duplicated(keep=False)
].sort_values(serial_col)

display(duplicate_serials[[
    "Tissue_ID", "Tissue_Serial", "Eye",
    "Image_ID", TARGET
]])

Rows: 1036
Unique serials: 1036
Duplicate serial rows: 0


,Tissue_ID,Tissue_Serial,Eye,Image_ID,Tissue_Grade_Clean


In [12]:
# Look for left/right records sharing the same serial or ID prefix.
id_check = labelled_df[["Tissue_ID", "Tissue_Serial", "Eye", TARGET]].copy()

id_check["Tissue_ID"] = id_check["Tissue_ID"].astype("string").str.strip()

id_check["ID_without_eye_suffix"] = (
    id_check["Tissue_ID"]
    .str.replace(r"\s+[RL](?:\s+\(.*\))?$", "", regex=True)
)

prefix_counts = (
    id_check.groupby("ID_without_eye_suffix", dropna=False)
    .agg(
        record_count=("Tissue_ID", "size"),
        unique_eyes=("Eye", "nunique"),
        unique_tissue_ids=("Tissue_ID", "nunique"),
    )
    .sort_values("record_count", ascending=False)
)

display(prefix_counts.head(20))

,record_count,unique_eyes,unique_tissue_ids
ID_without_eye_suffix,,,
NEB-2020-09-12654,1,1,1
NEB-2020-09-12655,1,1,1
NEB-2020-09-12656,1,1,1
NEB-2020-09-12657,1,1,1
NEB-2020-09-12658,1,1,1
NEB-2020-09-12659,1,1,1
NEB-2020-09-12660,1,1,1
NEB-2020-09-12661,1,1,1
NEB-2020-09-12662,1,1,1


In [13]:
grouping_candidates = [
    "Tissue_ID",
    "Tissue_Serial",
    "Image_ID",
    "Donor_Age",
    "Donor_Sex",
]

for col in grouping_candidates:
    if col in labelled_df.columns:
        print(
            f"{col}: {labelled_df[col].nunique(dropna=True)} unique, "
            f"{labelled_df[col].isna().sum()} missing"
        )

Tissue_ID: 1036 unique, 0 missing
Tissue_Serial: 1036 unique, 0 missing
Image_ID: 999 unique, 37 missing
Donor_Age: 57 unique, 2 missing
Donor_Sex: 21 unique, 478 missing


In [14]:
# 1. Inspect duplicated tissue serials
duplicate_serials = labelled_df[
    labelled_df["Tissue_Serial"].duplicated(keep=False)
].sort_values("Tissue_Serial")

print("Duplicated serial records:")
display(
    duplicate_serials[
        ["Tissue_ID", "Tissue_Serial", "Eye", "Image_ID", TARGET]
    ]
)

# 2. Check whether removing the eye suffix reveals repeated IDs
id_check = labelled_df[["Tissue_ID", "Eye", TARGET]].copy()

id_check["ID_without_eye_suffix"] = (
    id_check["Tissue_ID"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+[RL](?:\s+\(.*\))?$", "", regex=True)
)

repeated_prefixes = (
    id_check.groupby("ID_without_eye_suffix")
    .agg(
        records=("Tissue_ID", "size"),
        eyes=("Eye", "nunique"),
    )
    .query("records > 1")
    .sort_values("records", ascending=False)
)

print("Repeated ID prefixes:", len(repeated_prefixes))
display(repeated_prefixes.head(20))

Duplicated serial records:


,Tissue_ID,Tissue_Serial,Eye,Image_ID,Tissue_Grade_Clean


Repeated ID prefixes: 0


,records,eyes
ID_without_eye_suffix,,


In [15]:

# Check whether any available columns can identify the same donor
# across multiple records.

grouping_candidates = [
    "Tissue_ID",
    "Tissue_Serial",
    "Image_ID",
    "Donor_Age",
    "Donor_Sex",
]

for col in grouping_candidates:
    if col in labelled_df.columns:
        print(f"\n--- {col} ---")
        print("Non-null:", labelled_df[col].notna().sum())
        print("Unique:", labelled_df[col].nunique(dropna=True))
        print("Repeated values:", labelled_df[col].duplicated(keep=False).sum())

print("\nPotential donor-identification columns:")
print([
    col for col in labelled_df.columns
    if "donor" in col.lower()
    and any(term in col.lower() for term in ["id", "number", "code"])
])



--- Tissue_ID ---
Non-null: 1036
Unique: 1036
Repeated values: 0

--- Tissue_Serial ---
Non-null: 1036
Unique: 1036
Repeated values: 0

--- Image_ID ---
Non-null: 999
Unique: 999
Repeated values: 37

--- Donor_Age ---
Non-null: 1034
Unique: 57
Repeated values: 1029

--- Donor_Sex ---
Non-null: 558
Unique: 21
Repeated values: 1020

Potential donor-identification columns:
[]


In [16]:

# Define candidate feature groups for controlled model comparisons.

numeric_features = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
    "Donor_Age",
]

clinical_features = [
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Has_Scar",
    "Stroma_Arcus_Severity",
    "Stroma_Scar_Type",
    "Descemet_Has_Folds",
    "Descemet_Fold_Severity",
    "Descemet_Fold_Relative_Number",
    "Endo_Has_Stress_Lines",
    "Endo_Stress_Line_Location",
    "Endo_Stress_Line_Amount",
    "Endo_Has_Cell_Dropout",
    "Endo_Cell_Dropout_Severity",
    "Endo_Has_RBC",
    "Endo_RBC_Severity",
    "Epi_Has_Exposure",
    "Epi_Exposure_Severity",
    "Epi_Exposure_Location",
    "Epi_Has_Defect",
    "Epi_Has_Haze",
    "Epi_Haze_Severity",
    "Epi_Has_Arcus",
]

# ECD will be evaluated separately as a baseline and as a feature.
ecd_feature = ["ECD_CD_cells_per_mm2"]

numeric_without_ecd = [
    col for col in numeric_features if col not in ecd_feature
]

# Confirm all candidate features exist.
for group_name, features in {
    "Numeric": numeric_features,
    "Clinical": clinical_features,
    "Numeric without ECD": numeric_without_ecd,
}.items():
    missing = sorted(set(features) - set(labelled_df.columns))
    print(f"{group_name}: {len(features)} features")
    print("Missing columns:", missing)


Numeric: 7 features
Missing columns: []
Clinical: 24 features
Missing columns: []
Numeric without ECD: 6 features
Missing columns: []


In [19]:

# Audit candidate feature types and missingness.

candidate_features = numeric_features + clinical_features

feature_audit = pd.DataFrame({
    "Feature": candidate_features,
    "Data_Type": [
        str(labelled_df[col].dtype)
        for col in candidate_features
    ],
    "Non_Missing": [
        labelled_df[col].notna().sum()
        for col in candidate_features
    ],
    "Missing": [
        labelled_df[col].isna().sum()
        for col in candidate_features
    ],
    "Missing_Percent": [
        round(labelled_df[col].isna().mean() * 100, 2)
        for col in candidate_features
    ],
    "Unique_Values": [
        labelled_df[col].nunique(dropna=True)
        for col in candidate_features
    ],
})

print(display(
    feature_audit.sort_values(
        ["Missing_Percent", "Feature"],
        ascending=[False, True]
    ).reset_index(drop=True)
))


,Feature,Data_Type,Non_Missing,Missing,Missing_Percent,Unique_Values
0,Epi_Haze_Severity,str,29,1007,97.20,2
1,Endo_RBC_Severity,str,30,1006,97.10,3
2,Stroma_Scar_Type,str,33,1003,96.81,2
3,Endo_Cell_Dropout_Severity,str,91,945,91.22,2
4,Epi_Exposure_Location,str,307,729,70.37,4
5,Epi_Exposure_Severity,str,307,729,70.37,3
6,Stroma_Arcus_Severity,str,475,561,54.15,3
7,Endo_Stress_Line_Amount,str,488,548,52.90,3
8,Endo_Stress_Line_Location,str,501,535,51.64,4
9,Descemet_Fold_Relative_Number,str,529,507,48.94,5


None


In [20]:

# Inspect the recorded values of clinical features.
# This does not modify the dataset.

clinical_value_summary = []

for col in clinical_features:
    values = labelled_df[col].dropna()

    clinical_value_summary.append({
        "Feature": col,
        "Data_Type": str(labelled_df[col].dtype),
        "Recorded": len(values),
        "Unique_Values": values.nunique(),
        "Examples": values.astype(str).unique()[:10].tolist(),
    })

clinical_value_summary = pd.DataFrame(clinical_value_summary)

display(
    clinical_value_summary.sort_values(
        ["Recorded", "Feature"],
        ascending=[True, True]
    ).reset_index(drop=True)
)


,Feature,Data_Type,Recorded,Unique_Values,Examples
0,Epi_Haze_Severity,str,29,2,"[Unspecified, Mild]"
1,Endo_RBC_Severity,str,30,3,"[Unspecified, ++, +]"
2,Stroma_Scar_Type,str,33,2,"[Other scar, Phaco scar]"
3,Endo_Cell_Dropout_Severity,str,91,2,"[Mild, Moderate]"
4,Epi_Exposure_Location,str,307,4,"[Peripheral, Unspecified, Central, Central + P..."
5,Epi_Exposure_Severity,str,307,3,"[Mild, Unspecified, Very mild]"
6,Stroma_Arcus_Severity,str,475,3,"[Unspecified, Mild, Moderate]"
7,Endo_Stress_Line_Amount,str,488,3,"[Few, Unspecified, Several]"
8,Endo_Stress_Line_Location,str,501,4,"[Central, Peripheral, Unspecified, Central + P..."
9,Descemet_Fold_Relative_Number,str,529,5,"[Unspecified, Several, Mild, Moderate, Several..."


In [21]:

# Inspect binary clinical values and their missingness.
# This cell does not modify the dataset.

binary_clinical_features = [
    col for col in clinical_features
    if labelled_df[col].dropna().isin([0, 1, 0.0, 1.0]).all()
    and labelled_df[col].notna().any()
]

binary_summary = []

for col in binary_clinical_features:
    counts = labelled_df[col].value_counts(dropna=False)

    binary_summary.append({
        "Feature": col,
        "Present (1)": int((labelled_df[col] == 1).sum()),
        "Absent (0)": int((labelled_df[col] == 0).sum()),
        "Missing": int(labelled_df[col].isna().sum()),
    })

display(
    pd.DataFrame(binary_summary)
    .sort_values("Feature")
    .reset_index(drop=True)
)


,Feature,Present (1),Absent (0),Missing
0,Descemet_Has_Folds,523,6,507
1,Endo_Has_Cell_Dropout,91,470,475
2,Endo_Has_RBC,30,531,475
3,Endo_Has_Stress_Lines,488,73,475
4,Epi_Has_Arcus,326,246,464
5,Epi_Has_Defect,4,568,464
6,Epi_Has_Exposure,307,265,464
7,Epi_Has_Haze,29,543,464
8,Stroma_Has_Arcus,475,98,463
9,Stroma_Has_Edema,10,563,463


In [22]:
# Print the recorded categories for clinical fields.
# This cell does not modify the dataset.

categorical_clinical_features = [
    col for col in clinical_features
    if labelled_df[col].dtype == "str"
]

for col in categorical_clinical_features:
    print(f"\n--- {col} ---")
    print("Missing:", labelled_df[col].isna().sum())
    print(labelled_df[col].value_counts(dropna=False).to_string())


--- Stroma_Arcus_Severity ---
Missing: 561
Stroma_Arcus_Severity
NaN            561
Unspecified    432
Mild            38
Moderate         5

--- Stroma_Scar_Type ---
Missing: 1003
Stroma_Scar_Type
NaN           1003
Other scar      27
Phaco scar       6

--- Descemet_Fold_Severity ---
Missing: 507
Descemet_Fold_Severity
NaN            507
Mild           384
Unspecified    107
Moderate        21
Very mild       17

--- Descemet_Fold_Relative_Number ---
Missing: 507
Descemet_Fold_Relative_Number
NaN                 507
Unspecified         249
Mild                249
Several              28
Moderate              2
Several/Moderate      1

--- Endo_Stress_Line_Location ---
Missing: 535
Endo_Stress_Line_Location
NaN                     535
Central                 237
Unspecified             174
Peripheral               63
Central + Peripheral     27

--- Endo_Stress_Line_Amount ---
Missing: 548
Endo_Stress_Line_Amount
NaN            548
Few            468
Unspecified     18
Several       

In [23]:
# Check consistency between clinical flags and their related details.
# This cell does not modify the dataset.

clinical_pairs = {
    "Descemet_Has_Folds": [
        "Descemet_Fold_Severity",
        "Descemet_Fold_Relative_Number",
    ],
    "Endo_Has_Stress_Lines": [
        "Endo_Stress_Line_Location",
        "Endo_Stress_Line_Amount",
    ],
    "Endo_Has_Cell_Dropout": [
        "Endo_Cell_Dropout_Severity",
    ],
    "Endo_Has_RBC": [
        "Endo_RBC_Severity",
    ],
    "Epi_Has_Exposure": [
        "Epi_Exposure_Severity",
        "Epi_Exposure_Location",
    ],
    "Epi_Has_Haze": [
        "Epi_Haze_Severity",
    ],
    "Stroma_Has_Arcus": [
        "Stroma_Arcus_Severity",
    ],
    "Stroma_Has_Scar": [
        "Stroma_Scar_Type",
    ],
}

for flag, detail_columns in clinical_pairs.items():
    print(f"\n--- {flag} ---")

    for detail in detail_columns:
        recorded_detail = labelled_df[detail].notna()
        flag_missing = labelled_df[flag].isna()
        flag_zero = labelled_df[flag].eq(0)
        flag_one = labelled_df[flag].eq(1)

        print(f"\nDetail: {detail}")
        print("Detail recorded:", int(recorded_detail.sum()))
        print("Detail recorded while flag = 0:",
              int((recorded_detail & flag_zero).sum()))
        print("Detail recorded while flag = 1:",
              int((recorded_detail & flag_one).sum()))
        print("Detail recorded while flag is missing:",
              int((recorded_detail & flag_missing).sum()))


--- Descemet_Has_Folds ---

Detail: Descemet_Fold_Severity
Detail recorded: 529
Detail recorded while flag = 0: 6
Detail recorded while flag = 1: 523
Detail recorded while flag is missing: 0

Detail: Descemet_Fold_Relative_Number
Detail recorded: 529
Detail recorded while flag = 0: 6
Detail recorded while flag = 1: 523
Detail recorded while flag is missing: 0

--- Endo_Has_Stress_Lines ---

Detail: Endo_Stress_Line_Location
Detail recorded: 501
Detail recorded while flag = 0: 13
Detail recorded while flag = 1: 488
Detail recorded while flag is missing: 0

Detail: Endo_Stress_Line_Amount
Detail recorded: 488
Detail recorded while flag = 0: 0
Detail recorded while flag = 1: 488
Detail recorded while flag is missing: 0

--- Endo_Has_Cell_Dropout ---

Detail: Endo_Cell_Dropout_Severity
Detail recorded: 91
Detail recorded while flag = 0: 0
Detail recorded while flag = 1: 91
Detail recorded while flag is missing: 0

--- Endo_Has_RBC ---

Detail: Endo_RBC_Severity
Detail recorded: 30
Detail 

In [24]:

# Inspect clinical detail fields recorded when the binary flag is 0.
# This cell does not modify the dataset.

checks = {
    "Descemet_Has_Folds": [
        "Descemet_Fold_Severity",
        "Descemet_Fold_Relative_Number",
    ],
    "Endo_Has_Stress_Lines": [
        "Endo_Stress_Line_Location",
        "Endo_Stress_Line_Amount",
    ],
}

for flag, detail_columns in checks.items():
    mask = labelled_df[flag].eq(0) & labelled_df[detail_columns].notna().any(axis=1)

    print(f"\n{'=' * 60}")
    print(f"Flag: {flag}")
    print("Rows to inspect:", int(mask.sum()))

    if mask.any():
        columns_to_show = [
            "Tissue_ID",
            "Image_ID",
            TARGET,
            flag,
            *detail_columns,
        ]

        print(labelled_df.loc[mask, columns_to_show].to_string(index=False))
    else:
        print("No potential inconsistencies found.")



Flag: Descemet_Has_Folds
Rows to inspect: 6
          Tissue_ID               Image_ID Tissue_Grade_Clean  Descemet_Has_Folds Descemet_Fold_Severity Descemet_Fold_Relative_Number
NEB-2022-04-13824 R NEB-2022-04-13824 R_SP                  B                 0.0            Unspecified                   Unspecified
NEB-2022-10-14330 L NEB-2022-10-14330 L_SP                 A+                 0.0            Unspecified                   Unspecified
NEB-2022-10-14336 R NEB-2022-10-14336 R_SP                 A+                 0.0            Unspecified                   Unspecified
NEB-2024-12-16603 R NEB-2024-12-16603 R_SP                  A                 0.0                   Mild                          Mild
NEB-2025-11-17623 R NEB-2025-11-17623 R_SP                  A                 0.0            Unspecified                   Unspecified
NEB-2025-11-17624 L NEB-2025-11-17624 L_SP                  A                 0.0            Unspecified                   Unspecified

Flag: End

In [25]:

# Summarize potential conflicts between binary flags and clinical details.
# This cell does not modify the dataset.

review_cases = [
    (
        "Descemet_Has_Folds",
        ["Descemet_Fold_Severity", "Descemet_Fold_Relative_Number"],
    ),
    (
        "Endo_Has_Stress_Lines",
        ["Endo_Stress_Line_Location", "Endo_Stress_Line_Amount"],
    ),
]

for flag, details in review_cases:
    mask = labelled_df[flag].eq(0) & labelled_df[details].notna().any(axis=1)
    subset = labelled_df.loc[mask, [flag] + details]

    print(f"\n--- {flag}: {len(subset)} potential conflicts ---")

    for column in details:
        print(f"\n{column} values:")
        print(subset[column].value_counts(dropna=False).to_string())

    print("\nCombined detail patterns:")
    print(
        subset[details]
        .fillna("<MISSING>")
        .value_counts(dropna=False)
        .to_string()
    )



--- Descemet_Has_Folds: 6 potential conflicts ---

Descemet_Fold_Severity values:
Descemet_Fold_Severity
Unspecified    5
Mild           1

Descemet_Fold_Relative_Number values:
Descemet_Fold_Relative_Number
Unspecified    5
Mild           1

Combined detail patterns:
Descemet_Fold_Severity  Descemet_Fold_Relative_Number
Unspecified             Unspecified                      5
Mild                    Mild                             1

--- Endo_Has_Stress_Lines: 13 potential conflicts ---

Endo_Stress_Line_Location values:
Endo_Stress_Line_Location
Unspecified             6
Central                 6
Central + Peripheral    1

Endo_Stress_Line_Amount values:
Endo_Stress_Line_Amount
NaN    13

Combined detail patterns:
Endo_Stress_Line_Location  Endo_Stress_Line_Amount
Unspecified                <MISSING>                  6
Central                    <MISSING>                  6
Central + Peripheral       <MISSING>                  1


In [26]:

# Review clinical flag counts by specialist-reviewed tissue grade.
# Read-only: no changes to the dataset.

flags_to_review = [
    "Descemet_Has_Folds",
    "Endo_Has_Stress_Lines",
    "Endo_Has_Cell_Dropout",
    "Endo_Has_RBC",
    "Epi_Has_Exposure",
    "Epi_Has_Haze",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Has_Scar",
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
]

for flag in flags_to_review:
    print(f"\n--- {flag} by {TARGET} ---")

    counts = pd.crosstab(
        labelled_df[TARGET],
        labelled_df[flag].fillna(-1).map({
            -1: "Missing",
            0: "Absent",
            1: "Present",
        }),
        dropna=False,
    )

    print(counts.to_string())



--- Descemet_Has_Folds by Tissue_Grade_Clean ---
Descemet_Has_Folds  Absent  Missing  Present
Tissue_Grade_Clean                          
A                        3      137      179
A+                       2      275      268
B                        1       95       76

--- Endo_Has_Stress_Lines by Tissue_Grade_Clean ---
Endo_Has_Stress_Lines  Absent  Missing  Present
Tissue_Grade_Clean                             
A                          34      120      165
A+                         26      262      257
B                          13       93       66

--- Endo_Has_Cell_Dropout by Tissue_Grade_Clean ---
Endo_Has_Cell_Dropout  Absent  Missing  Present
Tissue_Grade_Clean                             
A                         165      120       34
A+                        240      262       43
B                          65       93       14

--- Endo_Has_RBC by Tissue_Grade_Clean ---
Endo_Has_RBC        Absent  Missing  Present
Tissue_Grade_Clean                          
A    

In [27]:

# Summarize clinical feature coverage.
# Read-only: this cell does not modify the dataset.

coverage_rows = []

for feature in clinical_features:
    recorded = labelled_df[feature].notna().sum()
    missing = labelled_df[feature].isna().sum()

    coverage_rows.append({
        "Feature": feature,
        "Recorded": int(recorded),
        "Missing": int(missing),
        "Coverage_pct": round(recorded / len(labelled_df) * 100, 2),
        "Unique_recorded_values": int(labelled_df[feature].nunique()),
    })

coverage_df = pd.DataFrame(coverage_rows)
coverage_df = coverage_df.sort_values(
    by="Coverage_pct",
    ascending=False,
)

print("Total labelled rows:", len(labelled_df))
print("\nClinical feature coverage:")
print(coverage_df.to_string(index=False))


Total labelled rows: 1036

Clinical feature coverage:
                      Feature  Recorded  Missing  Coverage_pct  Unique_recorded_values
              Stroma_Is_Clear       573      463         55.31                       2
            Stroma_Is_Compact       573      463         55.31                       2
             Stroma_Has_Arcus       573      463         55.31                       2
             Stroma_Has_Edema       573      463         55.31                       2
              Stroma_Has_Scar       573      463         55.31                       2
                Epi_Has_Arcus       572      464         55.21                       2
                 Epi_Has_Haze       572      464         55.21                       2
               Epi_Has_Defect       572      464         55.21                       2
             Epi_Has_Exposure       572      464         55.21                       2
        Endo_Has_Stress_Lines       561      475         54.15              

In [28]:

# Proposed clinical feature groups for initial experiments.
# These lists do not modify labelled_df.

clinical_binary_candidates = [
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Has_Scar",
    "Epi_Has_Arcus",
    "Epi_Has_Defect",
    "Epi_Has_Exposure",
    "Epi_Has_Haze",
    "Endo_Has_Stress_Lines",
    "Endo_Has_RBC",
    "Endo_Has_Cell_Dropout",
    "Descemet_Has_Folds",
]

clinical_categorical_candidates = [
    "Descemet_Fold_Severity",
    "Descemet_Fold_Relative_Number",
    "Endo_Stress_Line_Location",
    "Endo_Stress_Line_Amount",
    "Stroma_Arcus_Severity",
    "Epi_Exposure_Location",
    "Epi_Exposure_Severity",
]

clinical_sparse_review = [
    "Endo_Cell_Dropout_Severity",
    "Stroma_Scar_Type",
    "Endo_RBC_Severity",
    "Epi_Haze_Severity",
]

print("Binary clinical candidates:", len(clinical_binary_candidates))
print(*clinical_binary_candidates, sep="\n  ")

print("\nCategorical clinical candidates:", len(clinical_categorical_candidates))
print(*clinical_categorical_candidates, sep="\n  ")

print("\nSparse features for later review:", len(clinical_sparse_review))
print(*clinical_sparse_review, sep="\n  ")

all_selected = (
    clinical_binary_candidates
    + clinical_categorical_candidates
    + clinical_sparse_review
)

missing_columns = [
    column for column in all_selected
    if column not in labelled_df.columns
]

print("\nColumns not found in dataset:", missing_columns)
print("Unique selected columns:", len(set(all_selected)))


Binary clinical candidates: 13
Stroma_Is_Clear
  Stroma_Is_Compact
  Stroma_Has_Arcus
  Stroma_Has_Edema
  Stroma_Has_Scar
  Epi_Has_Arcus
  Epi_Has_Defect
  Epi_Has_Exposure
  Epi_Has_Haze
  Endo_Has_Stress_Lines
  Endo_Has_RBC
  Endo_Has_Cell_Dropout
  Descemet_Has_Folds

Categorical clinical candidates: 7
Descemet_Fold_Severity
  Descemet_Fold_Relative_Number
  Endo_Stress_Line_Location
  Endo_Stress_Line_Amount
  Stroma_Arcus_Severity
  Epi_Exposure_Location
  Epi_Exposure_Severity

Sparse features for later review: 4
Endo_Cell_Dropout_Severity
  Stroma_Scar_Type
  Endo_RBC_Severity
  Epi_Haze_Severity

Columns not found in dataset: []
Unique selected columns: 24


In [29]:

# Audit values before preprocessing.
# Read-only: this cell does not modify labelled_df.

feature_groups = {
    "Binary clinical candidates": clinical_binary_candidates,
    "Categorical clinical candidates": clinical_categorical_candidates,
    "Sparse features for later review": clinical_sparse_review,
}

for group_name, features in feature_groups.items():
    print(f"\n{'=' * 70}")
    print(group_name)
    print("=" * 70)

    for feature in features:
        values = labelled_df[feature].value_counts(dropna=False)

        print(f"\n{feature}")
        print("Distinct values including missing:", len(values))
        print(values.to_string())



Binary clinical candidates

Stroma_Is_Clear
Distinct values including missing: 3
Stroma_Is_Clear
1.0    564
NaN    463
0.0      9

Stroma_Is_Compact
Distinct values including missing: 3
Stroma_Is_Compact
NaN    463
0.0    344
1.0    229

Stroma_Has_Arcus
Distinct values including missing: 3
Stroma_Has_Arcus
1.0    475
NaN    463
0.0     98

Stroma_Has_Edema
Distinct values including missing: 3
Stroma_Has_Edema
0.0    563
NaN    463
1.0     10

Stroma_Has_Scar
Distinct values including missing: 3
Stroma_Has_Scar
0.0    540
NaN    463
1.0     33

Epi_Has_Arcus
Distinct values including missing: 3
Epi_Has_Arcus
NaN    464
1.0    326
0.0    246

Epi_Has_Defect
Distinct values including missing: 3
Epi_Has_Defect
0.0    568
NaN    464
1.0      4

Epi_Has_Exposure
Distinct values including missing: 3
Epi_Has_Exposure
NaN    464
1.0    307
0.0    265

Epi_Has_Haze
Distinct values including missing: 3
Epi_Has_Haze
0.0    543
NaN    464
1.0     29

Endo_Has_Stress_Lines
Distinct values includin

In [30]:

# Validate values in binary clinical candidates.
# Read-only: no changes to labelled_df.

for feature in clinical_binary_candidates:
    print(f"\n--- {feature} ---")
    print(
        labelled_df[feature]
        .value_counts(dropna=False)
        .to_string()
    )

    unexpected = labelled_df.loc[
        labelled_df[feature].notna()
        & ~labelled_df[feature].isin([0, 1]),
        feature,
    ]

    print("Unexpected non-binary values:", len(unexpected))
    if not unexpected.empty:
        print(unexpected.value_counts().to_string())



--- Stroma_Is_Clear ---
Stroma_Is_Clear
1.0    564
NaN    463
0.0      9
Unexpected non-binary values: 0

--- Stroma_Is_Compact ---
Stroma_Is_Compact
NaN    463
0.0    344
1.0    229
Unexpected non-binary values: 0

--- Stroma_Has_Arcus ---
Stroma_Has_Arcus
1.0    475
NaN    463
0.0     98
Unexpected non-binary values: 0

--- Stroma_Has_Edema ---
Stroma_Has_Edema
0.0    563
NaN    463
1.0     10
Unexpected non-binary values: 0

--- Stroma_Has_Scar ---
Stroma_Has_Scar
0.0    540
NaN    463
1.0     33
Unexpected non-binary values: 0

--- Epi_Has_Arcus ---
Epi_Has_Arcus
NaN    464
1.0    326
0.0    246
Unexpected non-binary values: 0

--- Epi_Has_Defect ---
Epi_Has_Defect
0.0    568
NaN    464
1.0      4
Unexpected non-binary values: 0

--- Epi_Has_Exposure ---
Epi_Has_Exposure
NaN    464
1.0    307
0.0    265
Unexpected non-binary values: 0

--- Epi_Has_Haze ---
Epi_Has_Haze
0.0    543
NaN    464
1.0     29
Unexpected non-binary values: 0

--- Endo_Has_Stress_Lines ---
Endo_Has_Stress_L

In [31]:

# Validate categories for the clinical categorical candidates.
# Read-only: no changes to labelled_df.

for feature in clinical_categorical_candidates:
    print(f"\n--- {feature} ---")

    counts = labelled_df[feature].value_counts(dropna=False)
    print(counts.to_string())

    recorded = int(labelled_df[feature].notna().sum())
    missing = int(labelled_df[feature].isna().sum())

    print("Recorded:", recorded)
    print("Missing:", missing)
    print("Distinct non-missing categories:",
          int(labelled_df[feature].nunique(dropna=True)))



--- Descemet_Fold_Severity ---
Descemet_Fold_Severity
NaN            507
Mild           384
Unspecified    107
Moderate        21
Very mild       17
Recorded: 529
Missing: 507
Distinct non-missing categories: 4

--- Descemet_Fold_Relative_Number ---
Descemet_Fold_Relative_Number
NaN                 507
Unspecified         249
Mild                249
Several              28
Moderate              2
Several/Moderate      1
Recorded: 529
Missing: 507
Distinct non-missing categories: 5

--- Endo_Stress_Line_Location ---
Endo_Stress_Line_Location
NaN                     535
Central                 237
Unspecified             174
Peripheral               63
Central + Peripheral     27
Recorded: 501
Missing: 535
Distinct non-missing categories: 4

--- Endo_Stress_Line_Amount ---
Endo_Stress_Line_Amount
NaN            548
Few            468
Unspecified     18
Several          2
Recorded: 488
Missing: 548
Distinct non-missing categories: 3

--- Stroma_Arcus_Severity ---
Stroma_Arcus_Severity
Na

In [32]:

# Define numeric feature groups for the initial experiments.
# This cell only defines lists and checks their columns.

ecd_features = [
    "ECD_CD_cells_per_mm2",
]

morphology_features = [
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
]

numeric_with_ecd = ecd_features + morphology_features

numeric_with_age = numeric_with_ecd + ["Donor_Age"]

feature_sets = {
    "ECD-only baseline": ecd_features,
    "Morphology excluding ECD": morphology_features,
    "ECD + morphology": numeric_with_ecd,
    "ECD + morphology + donor age": numeric_with_age,
}

for name, features in feature_sets.items():
    missing_columns = [
        feature for feature in features
        if feature not in labelled_df.columns
    ]

    print(f"\n--- {name} ---")
    print("Features:", features)
    print("Feature count:", len(features))
    print("Missing columns:", missing_columns)

print("\nTarget distribution:")
print(labelled_df[TARGET].value_counts().to_string())



--- ECD-only baseline ---
Features: ['ECD_CD_cells_per_mm2']
Feature count: 1
Missing columns: []

--- Morphology excluding ECD ---
Features: ['CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2']
Feature count: 5
Missing columns: []

--- ECD + morphology ---
Features: ['ECD_CD_cells_per_mm2', 'CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2']
Feature count: 6
Missing columns: []

--- ECD + morphology + donor age ---
Features: ['ECD_CD_cells_per_mm2', 'CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2', 'Donor_Age']
Feature count: 7
Missing columns: []

Target distribution:
Tissue_Grade_Clean
A+    545
A     319
B     172


In [33]:

# Define a reusable numeric preprocessing pipeline.
# No model is trained in this cell.

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

print("Numeric preprocessing pipeline:")
print(numeric_preprocessor)

print("\nMissing-value counts by numeric feature set:")

for name, features in feature_sets.items():
    missing_counts = labelled_df[features].isna().sum()

    print(f"\n--- {name} ---")
    print(missing_counts.to_string())
    print("Rows with at least one missing feature:",
          int(labelled_df[features].isna().any(axis=1).sum()))
    print("Rows complete across all features:",
          int(labelled_df[features].notna().all(axis=1).sum()))

print("\nTarget values:")
print(sorted(labelled_df[TARGET].unique().tolist()))


Numeric preprocessing pipeline:
Pipeline(steps=[('imputer', SimpleImputer(strategy='median'))])

Missing-value counts by numeric feature set:

--- ECD-only baseline ---
ECD_CD_cells_per_mm2    96
Rows with at least one missing feature: 96
Rows complete across all features: 940

--- Morphology excluding ECD ---
CV                  96
HEX_percent         96
SD                  96
Total_Cell_Count    96
Total_Area_um2      96
Rows with at least one missing feature: 96
Rows complete across all features: 940

--- ECD + morphology ---
ECD_CD_cells_per_mm2    96
CV                      96
HEX_percent             96
SD                      96
Total_Cell_Count        96
Total_Area_um2          96
Rows with at least one missing feature: 96
Rows complete across all features: 940

--- ECD + morphology + donor age ---
ECD_CD_cells_per_mm2    96
CV                      96
HEX_percent             96
SD                      96
Total_Cell_Count        96
Total_Area_um2          96
Donor_Age            

In [34]:

# Define the clinical preprocessing pipeline.
# No model is trained and labelled_df is not modified.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

clinical_binary_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median",
                add_indicator=True,
            ),
        ),
    ]
)

clinical_categorical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent",
            ),
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
            ),
        ),
    ]
)

clinical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "binary",
            clinical_binary_preprocessor,
            clinical_binary_candidates,
        ),
        (
            "categorical",
            clinical_categorical_preprocessor,
            clinical_categorical_candidates,
        ),
    ],
    remainder="drop",
)

print("Clinical preprocessing pipeline:")
print(clinical_preprocessor)

print("\nBinary input features:", len(clinical_binary_candidates))
print("Categorical input features:", len(clinical_categorical_candidates))
print(
    "Total input features:",
    len(clinical_binary_candidates)
    + len(clinical_categorical_candidates),
)

print("\nSparse features excluded from this initial pipeline:")
print(*clinical_sparse_review, sep="\n")


Clinical preprocessing pipeline:
ColumnTransformer(transformers=[('binary',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median'))]),
                                 ['Stroma_Is_Clear', 'Stroma_Is_Compact',
                                  'Stroma_Has_Arcus', 'Stroma_Has_Edema',
                                  'Stroma_Has_Scar', 'Epi_Has_Arcus',
                                  'Epi_Has_Defect', 'Epi_Has_Exposure',
                                  'Epi_Has_Haze', 'Endo_Has_Stress_Lines',
                                  'Endo_Has_RBC', 'Endo_Has_Cell_Dropout',
                                  'Descemet_Has_Folds']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_freque

In [35]:

# Refine clinical preprocessing.
# Binary flags remain 0/1 after imputation.
# No model is trained and labelled_df is not modified.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

clinical_binary_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent",
                add_indicator=True,
            ),
        ),
    ]
)

clinical_categorical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(strategy="most_frequent"),
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)

clinical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "binary",
            clinical_binary_preprocessor,
            clinical_binary_candidates,
        ),
        (
            "categorical",
            clinical_categorical_preprocessor,
            clinical_categorical_candidates,
        ),
    ],
    remainder="drop",
)

print("Updated clinical preprocessing pipeline:")
print(clinical_preprocessor)
print("\nBinary features:", len(clinical_binary_candidates))
print("Categorical features:", len(clinical_categorical_candidates))


Updated clinical preprocessing pipeline:
ColumnTransformer(transformers=[('binary',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='most_frequent'))]),
                                 ['Stroma_Is_Clear', 'Stroma_Is_Compact',
                                  'Stroma_Has_Arcus', 'Stroma_Has_Edema',
                                  'Stroma_Has_Scar', 'Epi_Has_Arcus',
                                  'Epi_Has_Defect', 'Epi_Has_Exposure',
                                  'Epi_Has_Haze', 'Endo_Has_Stress_Lines',
                                  'Endo_Has_RBC', 'Endo_Has_Cell_Dropout',
                                  'Descemet_Has_Folds']),
                                ('categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strate

In [36]:

# Diagnostic check of transformed clinical features.
# This is not model evaluation.

X_clinical = labelled_df[
    clinical_binary_candidates + clinical_categorical_candidates
]

X_clinical_transformed = clinical_preprocessor.fit_transform(X_clinical)

print("Original clinical matrix shape:", X_clinical.shape)
print("Transformed clinical matrix shape:", X_clinical_transformed.shape)
print(
    "Transformed matrix contains NaN:",
    bool(pd.isna(X_clinical_transformed).any()),
)
print("Target rows:", len(labelled_df))
print("Target classes:", sorted(labelled_df[TARGET].unique().tolist()))


Original clinical matrix shape: (1036, 20)
Transformed clinical matrix shape: (1036, 52)
Transformed matrix contains NaN: False
Target rows: 1036
Target classes: ['A', 'A+', 'B']


In [37]:

# Combined numeric + clinical preprocessing.
# This defines the transformer; it does not train a model.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

combined_numeric_features = numeric_with_ecd
combined_binary_features = clinical_binary_candidates
combined_categorical_features = clinical_categorical_candidates

combined_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median", add_indicator=True),
            combined_numeric_features,
        ),
        (
            "clinical_binary",
            SimpleImputer(
                strategy="most_frequent",
                add_indicator=True,
            ),
            combined_binary_features,
        ),
        (
            "clinical_categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(strategy="most_frequent"),
                    ),
                    (
                        "encoder",
                        OneHotEncoder(handle_unknown="ignore"),
                    ),
                ]
            ),
            combined_categorical_features,
        ),
    ],
    remainder="drop",
)

print("Combined preprocessing pipeline:")
print(combined_preprocessor)

print("\nNumeric features:", len(combined_numeric_features))
print("Binary clinical features:", len(combined_binary_features))
print("Categorical clinical features:", len(combined_categorical_features))
print(
    "Total original input columns:",
    len(combined_numeric_features)
    + len(combined_binary_features)
    + len(combined_categorical_features),
)


Combined preprocessing pipeline:
ColumnTransformer(transformers=[('numeric',
                                 SimpleImputer(add_indicator=True,
                                               strategy='median'),
                                 ['ECD_CD_cells_per_mm2', 'CV', 'HEX_percent',
                                  'SD', 'Total_Cell_Count', 'Total_Area_um2']),
                                ('clinical_binary',
                                 SimpleImputer(add_indicator=True,
                                               strategy='most_frequent'),
                                 ['Stroma_Is_Clear', 'Stroma_Is_Compact',
                                  'Stroma_Has_Arcus', 'Stroma_Has_Edema',
                                  'Stroma_Has_Sc...
                                  'Descemet_Has_Folds']),
                                ('clinical_categorical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImput

In [38]:

# Diagnostic check of the combined preprocessing pipeline.
# This is not model evaluation.

combined_features = (
    combined_numeric_features
    + combined_binary_features
    + combined_categorical_features
)

X_combined = labelled_df[combined_features]
X_combined_transformed = combined_preprocessor.fit_transform(X_combined)

print("Original combined matrix shape:", X_combined.shape)
print("Transformed combined matrix shape:", X_combined_transformed.shape)
print(
    "Transformed matrix contains NaN:",
    bool(pd.isna(X_combined_transformed).any()),
)
print("Target rows:", len(labelled_df))
print("Target classes:", sorted(labelled_df[TARGET].unique().tolist()))


Original combined matrix shape: (1036, 26)
Transformed combined matrix shape: (1036, 64)
Transformed matrix contains NaN: False
Target rows: 1036
Target classes: ['A', 'A+', 'B']


In [39]:

# Check the modelling environment before defining evaluation splits.

import sklearn
from sklearn.model_selection import StratifiedKFold, train_test_split

print("scikit-learn version:", sklearn.__version__)
print("Training target size:", len(labelled_df))
print("\nTarget distribution:")
print(labelled_df[TARGET].value_counts().to_string())

print("\nTarget proportions:")
print(
    (labelled_df[TARGET].value_counts(normalize=True) * 100)
    .round(2)
    .to_string()
)

print("\nStratifiedKFold available:", StratifiedKFold is not None)
print("train_test_split available:", train_test_split is not None)


scikit-learn version: 1.9.1
Training target size: 1036

Target distribution:
Tissue_Grade_Clean
A+    545
A     319
B     172

Target proportions:
Tissue_Grade_Clean
A+    52.61
A     30.79
B     16.60

StratifiedKFold available: True
train_test_split available: True


In [40]:

# Create a reproducible stratified train/test split.
# No model or preprocessing transformer is fitted here.

from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

indices = labelled_df.index

train_indices, test_indices = train_test_split(
    indices,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=labelled_df[TARGET],
)

y_train = labelled_df.loc[train_indices, TARGET]
y_test = labelled_df.loc[test_indices, TARGET]

print("Random state:", RANDOM_STATE)
print("Training rows:", len(train_indices))
print("Test rows:", len(test_indices))

print("\nTraining grade counts:")
print(y_train.value_counts().to_string())

print("\nTest grade counts:")
print(y_test.value_counts().to_string())

print("\nTraining grade proportions (%):")
print((y_train.value_counts(normalize=True) * 100).round(2).to_string())

print("\nTest grade proportions (%):")
print((y_test.value_counts(normalize=True) * 100).round(2).to_string())

print("\nIndex overlap between train and test:",
      len(set(train_indices) & set(test_indices)))


Random state: 42
Training rows: 828
Test rows: 208

Training grade counts:
Tissue_Grade_Clean
A+    436
A     255
B     137

Test grade counts:
Tissue_Grade_Clean
A+    109
A      64
B      35

Training grade proportions (%):
Tissue_Grade_Clean
A+    52.66
A     30.80
B     16.55

Test grade proportions (%):
Tissue_Grade_Clean
A+    52.40
A     30.77
B     16.83

Index overlap between train and test: 0


In [41]:

# ECD-only baseline: preprocessing is fitted on training data only.

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline

ecd_baseline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

X_train_ecd = labelled_df.loc[train_indices, ecd_features]
X_test_ecd = labelled_df.loc[test_indices, ecd_features]

ecd_baseline.fit(X_train_ecd, y_train)
y_pred_ecd = ecd_baseline.predict(X_test_ecd)

print("ECD-only baseline results")
print("Accuracy:", round(accuracy_score(y_test, y_pred_ecd), 4))
print("Balanced accuracy:", round(balanced_accuracy_score(y_test, y_pred_ecd), 4))
print("Macro F1:", round(f1_score(y_test, y_pred_ecd, average="macro"), 4))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred_ecd,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

print("Confusion matrix (rows = actual; columns = predicted)")
print("Class order: A+, A, B")
print(confusion_matrix(y_test, y_pred_ecd, labels=["A+", "A", "B"]))


ECD-only baseline results
Accuracy: 0.6538
Balanced accuracy: 0.7002
Macro F1: 0.6816

Classification report:
              precision    recall  f1-score   support

          A+     0.8101    0.5872    0.6809       109
           A     0.4615    0.6562    0.5419        64
           B     0.7895    0.8571    0.8219        35

    accuracy                         0.6538       208
   macro avg     0.6870    0.7002    0.6816       208
weighted avg     0.6994    0.6538    0.6618       208

Confusion matrix (rows = actual; columns = predicted)
Class order: A+, A, B
[[64 45  0]
 [14 42  8]
 [ 1  4 30]]


In [42]:

# Morphology-only baseline: deliberately excludes ECD.
# Imputation is fitted on training data only.

morphology_baseline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

X_train_morph = labelled_df.loc[train_indices, morphology_features]
X_test_morph = labelled_df.loc[test_indices, morphology_features]

morphology_baseline.fit(X_train_morph, y_train)
y_pred_morph = morphology_baseline.predict(X_test_morph)

print("Morphology-only baseline results")
print("Features:", morphology_features)
print("Accuracy:", round(accuracy_score(y_test, y_pred_morph), 4))
print(
    "Balanced accuracy:",
    round(balanced_accuracy_score(y_test, y_pred_morph), 4),
)
print("Macro F1:", round(f1_score(y_test, y_pred_morph, average="macro"), 4))

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred_morph,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

print("\nConfusion matrix (rows = actual; columns = predicted)")
print("Class order: A+, A, B")
print(confusion_matrix(y_test, y_pred_morph, labels=["A+", "A", "B"]))


Morphology-only baseline results
Features: ['CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2']
Accuracy: 0.726
Balanced accuracy: 0.7589
Macro F1: 0.7515

Classification report:
              precision    recall  f1-score   support

          A+     0.8765    0.6514    0.7474       109
           A     0.5368    0.7969    0.6415        64
           B     0.9062    0.8286    0.8657        35

    accuracy                         0.7260       208
   macro avg     0.7732    0.7589    0.7515       208
weighted avg     0.7770    0.7260    0.7347       208


Confusion matrix (rows = actual; columns = predicted)
Class order: A+, A, B
[[71 38  0]
 [10 51  3]
 [ 0  6 29]]


C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [43]:

# Re-evaluate ECD-only and morphology-only baselines with scaling.
# Preprocessing is fitted on training data only.

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def make_numeric_baseline():
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "classifier",
                LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

baseline_results = {}

baseline_feature_sets = {
    "ECD only": ecd_features,
    "Morphology only": morphology_features,
}

for name, features in baseline_feature_sets.items():
    X_train = labelled_df.loc[train_indices, features]
    X_test = labelled_df.loc[test_indices, features]

    model = make_numeric_baseline()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    baseline_results[name] = {
        "model": model,
        "predictions": predictions,
        "accuracy": accuracy_score(y_test, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "macro_f1": f1_score(
            y_test,
            predictions,
            average="macro",
        ),
    }

    print(f"\n--- {name} ---")
    print("Accuracy:", round(baseline_results[name]["accuracy"], 4))
    print(
        "Balanced accuracy:",
        round(baseline_results[name]["balanced_accuracy"], 4),
    )
    print("Macro F1:", round(baseline_results[name]["macro_f1"], 4))
    print(
        "Iterations used:",
        model.named_steps["classifier"].n_iter_.tolist(),
    )

print("\n--- Comparison ---")
for metric in ["accuracy", "balanced_accuracy", "macro_f1"]:
    print(f"\n{metric}:")
    for name in baseline_results:
        print(
            f"  {name}: "
            f"{baseline_results[name][metric]:.4f}"
        )



--- ECD only ---
Accuracy: 0.6538
Balanced accuracy: 0.7002
Macro F1: 0.6816
Iterations used: [9]

--- Morphology only ---
Accuracy: 0.7404
Balanced accuracy: 0.7724
Macro F1: 0.7665
Iterations used: [26]

--- Comparison ---

accuracy:
  ECD only: 0.6538
  Morphology only: 0.7404

balanced_accuracy:
  ECD only: 0.7002
  Morphology only: 0.7724

macro_f1:
  ECD only: 0.6816
  Morphology only: 0.7665


In [44]:

# ECD + morphology baseline.
# Imputation and scaling are fitted on training data only.

combined_numeric_baseline = make_numeric_baseline()

X_train_numeric = labelled_df.loc[train_indices, numeric_with_ecd]
X_test_numeric = labelled_df.loc[test_indices, numeric_with_ecd]

combined_numeric_baseline.fit(X_train_numeric, y_train)
y_pred_numeric = combined_numeric_baseline.predict(X_test_numeric)

combined_numeric_results = {
    "accuracy": accuracy_score(y_test, y_pred_numeric),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred_numeric),
    "macro_f1": f1_score(
        y_test,
        y_pred_numeric,
        average="macro",
    ),
}

print("ECD + morphology results")
for metric, value in combined_numeric_results.items():
    print(f"{metric}: {value:.4f}")

print(
    "Iterations used:",
    combined_numeric_baseline.named_steps["classifier"].n_iter_.tolist(),
)

print("\nComparison with individual baselines:")
for name, results in baseline_results.items():
    print(f"\n{name}")
    print("Accuracy:", round(results["accuracy"], 4))
    print("Balanced accuracy:", round(results["balanced_accuracy"], 4))
    print("Macro F1:", round(results["macro_f1"], 4))

print("\nECD + morphology")
for metric, value in combined_numeric_results.items():
    print(f"{metric}: {value:.4f}")


ECD + morphology results
accuracy: 0.7404
balanced_accuracy: 0.7724
macro_f1: 0.7665
Iterations used: [23]

Comparison with individual baselines:

ECD only
Accuracy: 0.6538
Balanced accuracy: 0.7002
Macro F1: 0.6816

Morphology only
Accuracy: 0.7404
Balanced accuracy: 0.7724
Macro F1: 0.7665

ECD + morphology
accuracy: 0.7404
balanced_accuracy: 0.7724
macro_f1: 0.7665


In [45]:

# Compare predictions from morphology-only and ECD + morphology.
# Uses the same test rows for both fitted models.

morph_predictions = baseline_results["Morphology only"]["predictions"]
combined_predictions = y_pred_numeric

prediction_comparison = pd.DataFrame({
    "Actual": y_test,
    "Morphology_Prediction": morph_predictions,
    "ECD_Morphology_Prediction": combined_predictions,
})

changed = (
    prediction_comparison["Morphology_Prediction"]
    != prediction_comparison["ECD_Morphology_Prediction"]
)

print("Test rows:", len(prediction_comparison))
print("Predictions changed after adding ECD:", int(changed.sum()))
print(
    "Predictions unchanged:",
    int((~changed).sum()),
)

if changed.any():
    print("\nRows where predictions differ:")
    print(prediction_comparison.loc[changed].to_string())

print("\nCorrect predictions:")
print(
    "Morphology only:",
    int(
        (
            prediction_comparison["Morphology_Prediction"]
            == prediction_comparison["Actual"]
        ).sum(),
    ),
)
print(
    "ECD + morphology:",
    int(
        (
            prediction_comparison["ECD_Morphology_Prediction"]
            == prediction_comparison["Actual"]
        ).sum(),
    ),
)


Test rows: 208
Predictions changed after adding ECD: 0
Predictions unchanged: 208

Correct predictions:
Morphology only: 154
ECD + morphology: 154


## Preliminary Numeric Baseline Results

Three scaled logistic regression baselines were evaluated on the same stratified holdout test set of 208 samples. Median imputation and standardization were fitted using training data only, and class weights were balanced.

| Model | Accuracy | Balanced Accuracy | Macro F1 |
|---|---:|---:|---:|
| ECD only | 0.6538 | 0.7002 | 0.6816 |
| Morphology excluding ECD | 0.7404 | 0.7724 | 0.7665 |
| ECD + morphology | 0.7404 | 0.7724 | 0.7665 |

The morphology-only model outperformed the ECD-only baseline across all three metrics. Adding ECD to the morphology-only model did not change any of the 208 test predictions.

These findings are preliminary and apply to one row-level holdout split. Donor-independent generalization has not yet been established because a reliable donor identifier is unavailable. Further validation and experiments with clinical features are required before drawing final conclusions.

In [46]:

# Clinical-only baseline.
# All imputation and categorical encoding are fitted on training data only.

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline

clinical_baseline = Pipeline(
    steps=[
        ("preprocessor", clinical_preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

clinical_features = (
    clinical_binary_candidates
    + clinical_categorical_candidates
)

X_train_clinical = labelled_df.loc[
    train_indices, clinical_features
]
X_test_clinical = labelled_df.loc[
    test_indices, clinical_features
]

clinical_baseline.fit(X_train_clinical, y_train)

y_pred_clinical = clinical_baseline.predict(X_test_clinical)

clinical_results = {
    "accuracy": accuracy_score(y_test, y_pred_clinical),
    "balanced_accuracy": balanced_accuracy_score(
        y_test, y_pred_clinical
    ),
    "macro_f1": f1_score(
        y_test, y_pred_clinical, average="macro"
    ),
}

print("Clinical-only baseline results")
print("Original input features:", len(clinical_features))
print("Training rows:", len(X_train_clinical))
print("Test rows:", len(X_test_clinical))

for metric, value in clinical_results.items():
    print(f"{metric}: {value:.4f}")

print(
    "\nIterations used:",
    clinical_baseline.named_steps["classifier"].n_iter_.tolist(),
)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred_clinical,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

print("Confusion matrix (rows = actual; columns = predicted)")
print("Class order: A+, A, B")
print(
    confusion_matrix(
        y_test,
        y_pred_clinical,
        labels=["A+", "A", "B"],
    )
)


Clinical-only baseline results
Original input features: 20
Training rows: 828
Test rows: 208
accuracy: 0.3221
balanced_accuracy: 0.4074
macro_f1: 0.3224

Iterations used: [157]

Classification report:
              precision    recall  f1-score   support

          A+     0.5476    0.2110    0.3046       109
           A     0.2879    0.2969    0.2923        64
           B     0.2500    0.7143    0.3704        35

    accuracy                         0.3221       208
   macro avg     0.3618    0.4074    0.3224       208
weighted avg     0.4176    0.3221    0.3119       208

Confusion matrix (rows = actual; columns = predicted)
Class order: A+, A, B
[[23 43 43]
 [13 19 32]
 [ 6  4 25]]


In [47]:

# Preserve missing categorical values as their own category.
# All categories are learned/encoded within the training pipeline.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

clinical_binary_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent",
                add_indicator=True,
            ),
        ),
    ]
)

clinical_categorical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__",
            ),
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)

clinical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "binary",
            clinical_binary_preprocessor,
            clinical_binary_candidates,
        ),
        (
            "categorical",
            clinical_categorical_preprocessor,
            clinical_categorical_candidates,
        ),
    ],
    remainder="drop",
)

clinical_baseline = Pipeline(
    steps=[
        ("preprocessor", clinical_preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

clinical_baseline.fit(X_train_clinical, y_train)
y_pred_clinical = clinical_baseline.predict(X_test_clinical)

clinical_results = {
    "accuracy": accuracy_score(y_test, y_pred_clinical),
    "balanced_accuracy": balanced_accuracy_score(
        y_test, y_pred_clinical
    ),
    "macro_f1": f1_score(
        y_test, y_pred_clinical, average="macro"
    ),
}

print("Clinical-only baseline with explicit missing categories")
for metric, value in clinical_results.items():
    print(f"{metric}: {value:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred_clinical,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

print("Confusion matrix (rows = actual; columns = predicted)")
print(confusion_matrix(y_test, y_pred_clinical, labels=["A+", "A", "B"]))


Clinical-only baseline with explicit missing categories
accuracy: 0.3317
balanced_accuracy: 0.4157
macro_f1: 0.3321

Classification report:
              precision    recall  f1-score   support

          A+     0.5581    0.2202    0.3158       109
           A     0.3077    0.3125    0.3101        64
           B     0.2500    0.7143    0.3704        35

    accuracy                         0.3317       208
   macro avg     0.3719    0.4157    0.3321       208
weighted avg     0.4292    0.3317    0.3232       208

Confusion matrix (rows = actual; columns = predicted)
[[24 42 43]
 [12 20 32]
 [ 7  3 25]]


In [48]:

# Treat missing binary flags as a separate category.
# Preprocessing is fitted only on training data through the pipeline.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

clinical_binary_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value=-1,
            ),
        ),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore"),
        ),
    ]
)

clinical_preprocessor = ColumnTransformer(
    transformers=[
        (
            "binary",
            clinical_binary_preprocessor,
            clinical_binary_candidates,
        ),
        (
            "categorical",
            clinical_categorical_preprocessor,
            clinical_categorical_candidates,
        ),
    ],
    remainder="drop",
)

clinical_baseline = Pipeline(
    steps=[
        ("preprocessor", clinical_preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

clinical_baseline.fit(X_train_clinical, y_train)
y_pred_clinical = clinical_baseline.predict(X_test_clinical)

clinical_results = {
    "accuracy": accuracy_score(y_test, y_pred_clinical),
    "balanced_accuracy": balanced_accuracy_score(
        y_test, y_pred_clinical
    ),
    "macro_f1": f1_score(
        y_test, y_pred_clinical, average="macro"
    ),
}

print("Clinical-only baseline with explicit missing categories")
for metric, value in clinical_results.items():
    print(f"{metric}: {value:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred_clinical,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

print("Confusion matrix (rows = actual; columns = predicted)")
print(confusion_matrix(y_test, y_pred_clinical, labels=["A+", "A", "B"]))


Clinical-only baseline with explicit missing categories
accuracy: 0.3365
balanced_accuracy: 0.4166
macro_f1: 0.3358

Classification report:
              precision    recall  f1-score   support

          A+     0.5652    0.2385    0.3355       109
           A     0.3065    0.2969    0.3016        64
           B     0.2500    0.7143    0.3704        35

    accuracy                         0.3365       208
   macro avg     0.3739    0.4166    0.3358       208
weighted avg     0.4326    0.3365    0.3309       208

Confusion matrix (rows = actual; columns = predicted)
[[26 40 43]
 [13 19 32]
 [ 7  3 25]]


In [49]:

# Combined numeric + clinical baseline.
# All preprocessing is fitted on training data only.

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Numeric features: ECD + five morphology measurements
combined_numeric_features = numeric_with_ecd

# Clinical features: 13 binary flags + 7 categorical features
combined_binary_features = clinical_binary_candidates
combined_categorical_features = clinical_categorical_candidates

# Preserve missing binary values as a separate category (-1).
combined_binary_preprocessor = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

# Preserve missing categorical values as a separate category.
combined_categorical_preprocessor = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__",
            ),
        ),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

combined_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            combined_numeric_features,
        ),
        (
            "clinical_binary",
            combined_binary_preprocessor,
            combined_binary_features,
        ),
        (
            "clinical_categorical",
            combined_categorical_preprocessor,
            combined_categorical_features,
        ),
    ],
    remainder="drop",
)

combined_model = Pipeline(
    steps=[
        ("preprocessor", combined_preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

combined_features = (
    combined_numeric_features
    + combined_binary_features
    + combined_categorical_features
)

X_train_combined = labelled_df.loc[train_indices, combined_features]
X_test_combined = labelled_df.loc[test_indices, combined_features]

combined_model.fit(X_train_combined, y_train)
y_pred_combined = combined_model.predict(X_test_combined)

combined_results = {
    "accuracy": accuracy_score(y_test, y_pred_combined),
    "balanced_accuracy": balanced_accuracy_score(
        y_test, y_pred_combined
    ),
    "macro_f1": f1_score(
        y_test, y_pred_combined, average="macro"
    ),
}

print("Combined numeric + clinical baseline")
print("Original input features:", len(combined_features))
print("Training rows:", len(X_train_combined))
print("Test rows:", len(X_test_combined))

for metric, value in combined_results.items():
    print(f"{metric}: {value:.4f}")

print(
    "Iterations used:",
    combined_model.named_steps["classifier"].n_iter_.tolist(),
)

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred_combined,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

print("\nConfusion matrix (rows = actual; columns = predicted)")
print("Class order: A+, A, B")
print(
    confusion_matrix(
        y_test,
        y_pred_combined,
        labels=["A+", "A", "B"],
    )
)


Combined numeric + clinical baseline
Original input features: 26
Training rows: 828
Test rows: 208
accuracy: 0.7500
balanced_accuracy: 0.7678
macro_f1: 0.7585
Iterations used: [155]

Classification report:
              precision    recall  f1-score   support

          A+     0.8778    0.7248    0.7940       109
           A     0.5783    0.7500    0.6531        64
           B     0.8286    0.8286    0.8286        35

    accuracy                         0.7500       208
   macro avg     0.7616    0.7678    0.7585       208
weighted avg     0.7774    0.7500    0.7564       208


Confusion matrix (rows = actual; columns = predicted)
Class order: A+, A, B
[[79 30  0]
 [10 48  6]
 [ 1  5 29]]


In [50]:

import pandas as pd
import numpy as np

# Retrieve the fitted preprocessing and classifier steps.
fitted_preprocessor = combined_model.named_steps["preprocessor"]
fitted_classifier = combined_model.named_steps["classifier"]

# Get the feature names after imputation, scaling and one-hot encoding.
transformed_feature_names = fitted_preprocessor.get_feature_names_out()

# Logistic regression has one coefficient row per class.
coefficient_df = pd.DataFrame(
    fitted_classifier.coef_,
    columns=transformed_feature_names,
    index=fitted_classifier.classes_,
)

print("Transformed feature count:", len(transformed_feature_names))
print("Model classes:", fitted_classifier.classes_)
print("Coefficient matrix shape:", coefficient_df.shape)

print("\nFirst 15 transformed feature names:")
for feature_name in transformed_feature_names[:15]:
    print(feature_name)


Transformed feature count: 78
Model classes: ['A' 'A+' 'B']
Coefficient matrix shape: (3, 78)

First 15 transformed feature names:
numeric__ECD_CD_cells_per_mm2
numeric__CV
numeric__HEX_percent
numeric__SD
numeric__Total_Cell_Count
numeric__Total_Area_um2
clinical_binary__Stroma_Is_Clear_-1.0
clinical_binary__Stroma_Is_Clear_0.0
clinical_binary__Stroma_Is_Clear_1.0
clinical_binary__Stroma_Is_Compact_-1.0
clinical_binary__Stroma_Is_Compact_0.0
clinical_binary__Stroma_Is_Compact_1.0
clinical_binary__Stroma_Has_Arcus_-1.0
clinical_binary__Stroma_Has_Arcus_0.0
clinical_binary__Stroma_Has_Arcus_1.0


In [51]:

# Display the strongest positive and negative coefficients per class.
# Numeric features are standardized; one-hot-encoded features are not.

for grade in coefficient_df.index:
    grade_coefficients = coefficient_df.loc[grade].sort_values()

    print(f"\n{'=' * 55}")
    print(f"CLASS: {grade}")
    print("=" * 55)

    print("\nTop 10 positive coefficients:")
    print(grade_coefficients.tail(10).sort_values(ascending=False).to_string())

    print("\nTop 10 negative coefficients:")
    print(grade_coefficients.head(10).to_string())



CLASS: A

Top 10 positive coefficients:
numeric__Total_Area_um2                                                 1.248302
numeric__ECD_CD_cells_per_mm2                                           1.224933
clinical_categorical__Epi_Exposure_Location_Central + Peripheral        0.573530
clinical_categorical__Endo_Stress_Line_Amount_Few                       0.497476
clinical_categorical__Epi_Exposure_Location_Peripheral                  0.429102
clinical_binary__Endo_Has_Stress_Lines_0.0                              0.418340
clinical_categorical__Endo_Stress_Line_Location_Central + Peripheral    0.404129
clinical_binary__Descemet_Has_Folds_0.0                                 0.346613
clinical_categorical__Endo_Stress_Line_Amount___MISSING__               0.288229
clinical_categorical__Endo_Stress_Line_Location_Peripheral              0.283744

Top 10 negative coefficients:
clinical_categorical__Epi_Exposure_Location_Central                    -1.124579
clinical_categorical__Descemet_Fold_S

In [52]:

clinical_prefixes = (
    "clinical_binary__",
    "clinical_categorical__",
)

clinical_coefficient_df = coefficient_df.loc[
    :,
    coefficient_df.columns.str.startswith(clinical_prefixes),
]

for grade in clinical_coefficient_df.index:
    grade_coefficients = clinical_coefficient_df.loc[grade].sort_values()

    print(f"\n{'=' * 55}")
    print(f"CLINICAL COEFFICIENTS — CLASS {grade}")
    print("=" * 55)

    print("\nTop 10 positive clinical coefficients:")
    print(
        grade_coefficients.tail(10)
        .sort_values(ascending=False)
        .to_string()
    )

    print("\nTop 10 negative clinical coefficients:")
    print(grade_coefficients.head(10).to_string())



CLINICAL COEFFICIENTS — CLASS A

Top 10 positive clinical coefficients:
clinical_categorical__Epi_Exposure_Location_Central + Peripheral        0.573530
clinical_categorical__Endo_Stress_Line_Amount_Few                       0.497476
clinical_categorical__Epi_Exposure_Location_Peripheral                  0.429102
clinical_binary__Endo_Has_Stress_Lines_0.0                              0.418340
clinical_categorical__Endo_Stress_Line_Location_Central + Peripheral    0.404129
clinical_binary__Descemet_Has_Folds_0.0                                 0.346613
clinical_categorical__Endo_Stress_Line_Amount___MISSING__               0.288229
clinical_categorical__Endo_Stress_Line_Location_Peripheral              0.283744
clinical_categorical__Descemet_Fold_Severity_Moderate                   0.276871
clinical_categorical__Descemet_Fold_Severity_Unspecified                0.270700

Top 10 negative clinical coefficients:
clinical_categorical__Epi_Exposure_Location_Central                    -1.124

In [53]:

# Audit the support for clinical categories highlighted by the model.
# Counts are calculated across the labelled dataset, not just the test set.

clinical_categories_to_check = {
    "Descemet_Fold_Severity": [
        "Mild",
        "Very mild",
        "Moderate",
        "Unspecified",
    ],
    "Stroma_Has_Scar": [0, 1],
    "Epi_Exposure_Location": [
        "Central",
        "Peripheral",
        "Central + Peripheral",
        "Unspecified",
    ],
    "Endo_Stress_Line_Amount": [
        "Few",
        "Several",
        "Unspecified",
    ],
}

for column, categories in clinical_categories_to_check.items():
    print(f"\n{'=' * 55}")
    print(f"{column}")
    print("=" * 55)

    counts = labelled_df[column].value_counts(dropna=False)

    for category in categories:
        print(f"{category}: {counts.get(category, 0)}")

    print(f"Missing values: {labelled_df[column].isna().sum()}")



Descemet_Fold_Severity
Mild: 384
Very mild: 17
Moderate: 21
Unspecified: 107
Missing values: 507

Stroma_Has_Scar
0: 540
1: 33
Missing values: 463

Epi_Exposure_Location
Central: 5
Peripheral: 93
Central + Peripheral: 52
Unspecified: 157
Missing values: 729

Endo_Stress_Line_Amount
Few: 468
Several: 2
Unspecified: 18
Missing values: 548


In [54]:

# Cross-tabulate selected clinical categories against tissue grade.
# Missing values remain visible as their own category.

clinical_columns_to_audit = [
    "Descemet_Fold_Severity",
    "Stroma_Has_Scar",
    "Epi_Exposure_Location",
    "Endo_Stress_Line_Amount",
]

for column in clinical_columns_to_audit:
    print(f"\n{'=' * 65}")
    print(f"{column} by {TARGET}")
    print("=" * 65)

    audit_series = labelled_df[column].astype("object").fillna("__MISSING__")

    counts = pd.crosstab(
        audit_series,
        labelled_df[TARGET],
        dropna=False,
    )

    # Ensure the grade columns appear in a consistent order.
    counts = counts.reindex(columns=["A+", "A", "B"], fill_value=0)

    counts["Total"] = counts.sum(axis=1)

    print(counts.to_string())



Descemet_Fold_Severity by Tissue_Grade_Clean
Tissue_Grade_Clean       A+    A   B  Total
Descemet_Fold_Severity                     
Mild                    200  136  48    384
Moderate                  1    9  11     21
Unspecified              56   35  16    107
Very mild                13    2   2     17
__MISSING__             275  137  95    507

Stroma_Has_Scar by Tissue_Grade_Clean
Tissue_Grade_Clean   A+    A   B  Total
Stroma_Has_Scar                        
0.0                 289  188  63    540
1.0                   0   12  21     33
__MISSING__         256  119  88    463

Epi_Exposure_Location by Tissue_Grade_Clean
Tissue_Grade_Clean      A+    A    B  Total
Epi_Exposure_Location                      
Central                  2    1    2      5
Central + Peripheral    26   21    5     52
Peripheral              40   42   11     93
Unspecified             77   55   25    157
__MISSING__            400  200  129    729

Endo_Stress_Line_Amount by Tissue_Grade_Clean
Tissue_

In [55]:

# Training-set category counts for the clinical features under review.
# The test set is deliberately excluded from this diagnostic.

clinical_columns_to_audit = [
    "Descemet_Fold_Severity",
    "Stroma_Has_Scar",
    "Epi_Exposure_Location",
    "Endo_Stress_Line_Amount",
]

clinical_training_df = labelled_df.loc[train_indices]

for column in clinical_columns_to_audit:
    print(f"\n{'=' * 65}")
    print(f"TRAINING SET: {column} by {TARGET}")
    print("=" * 65)

    values = (
        clinical_training_df[column]
        .astype("object")
        .fillna("__MISSING__")
    )

    counts = pd.crosstab(values, clinical_training_df[TARGET])
    counts = counts.reindex(columns=["A+", "A", "B"], fill_value=0)
    counts["Total"] = counts.sum(axis=1)

    print(counts.to_string())



TRAINING SET: Descemet_Fold_Severity by Tissue_Grade_Clean
Tissue_Grade_Clean       A+    A   B  Total
Descemet_Fold_Severity                     
Mild                    152  111  35    298
Moderate                  1    7   8     16
Unspecified              45   26  15     86
Very mild                11    1   2     14
__MISSING__             227  110  77    414

TRAINING SET: Stroma_Has_Scar by Tissue_Grade_Clean
Tissue_Grade_Clean   A+    A   B  Total
Stroma_Has_Scar                        
0.0                 223  153  49    425
1.0                   0    9  16     25
__MISSING__         213   93  72    378

TRAINING SET: Epi_Exposure_Location by Tissue_Grade_Clean
Tissue_Grade_Clean      A+    A    B  Total
Epi_Exposure_Location                      
Central                  2    0    2      4
Central + Peripheral    18   18    3     39
Peripheral              34   35   11     80
Unspecified             61   45   18    124
__MISSING__            321  157  103    581

TRAINING SE

In [56]:

# Reduced clinical feature set for a sensitivity experiment.
# Exclude categories with extremely low support identified in the audit.

reduced_categorical_features = [
    column
    for column in clinical_categorical_candidates
    if column not in [
        "Epi_Exposure_Location",
        "Endo_Stress_Line_Amount",
    ]
]

reduced_clinical_features = (
    clinical_binary_candidates + reduced_categorical_features
)

print("Original clinical feature count:", len(clinical_features))
print("Reduced clinical feature count:", len(reduced_clinical_features))

print("\nExcluded categorical features:")
print("Epi_Exposure_Location")
print("Endo_Stress_Line_Amount")

print("\nRemaining categorical features:")
for feature in reduced_categorical_features:
    print(feature)


Original clinical feature count: 20
Reduced clinical feature count: 18

Excluded categorical features:
Epi_Exposure_Location
Endo_Stress_Line_Amount

Remaining categorical features:
Descemet_Fold_Severity
Descemet_Fold_Relative_Number
Endo_Stress_Line_Location
Stroma_Arcus_Severity
Epi_Exposure_Severity


In [57]:

# Reduced combined model: numeric + reduced clinical features

reduced_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_with_ecd,
        ),
        (
            "clinical_binary",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            clinical_binary_candidates,
        ),
        (
            "clinical_categorical",
            Pipeline(
                steps=[
                    (
                        "imputer",
                        SimpleImputer(
                            strategy="constant",
                            fill_value="__MISSING__",
                        ),
                    ),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            reduced_categorical_features,
        ),
    ],
    remainder="drop",
)

reduced_combined_model = Pipeline(
    steps=[
        ("preprocessor", reduced_preprocessor),
        (
            "classifier",
            LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

reduced_features = numeric_with_ecd + reduced_clinical_features

X_train_reduced = labelled_df.loc[train_indices, reduced_features]
X_test_reduced = labelled_df.loc[test_indices, reduced_features]

reduced_combined_model.fit(X_train_reduced, y_train)
y_pred_reduced = reduced_combined_model.predict(X_test_reduced)

reduced_results = {
    "accuracy": accuracy_score(y_test, y_pred_reduced),
    "balanced_accuracy": balanced_accuracy_score(y_test, y_pred_reduced),
    "macro_f1": f1_score(y_test, y_pred_reduced, average="macro"),
}

print("Reduced numeric + clinical baseline")
print("Original input features:", len(combined_features))
print("Reduced input features:", len(reduced_features))
print("Training rows:", len(X_train_reduced))
print("Test rows:", len(X_test_reduced))

for metric, value in reduced_results.items():
    print(f"{metric}: {value:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        y_pred_reduced,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

print("\nConfusion matrix (rows = actual; columns = predicted)")
print("Class order: A+, A, B")
print(
    confusion_matrix(
        y_test,
        y_pred_reduced,
        labels=["A+", "A", "B"],
    )
)


Reduced numeric + clinical baseline
Original input features: 26
Reduced input features: 24
Training rows: 828
Test rows: 208
accuracy: 0.7548
balanced_accuracy: 0.7708
macro_f1: 0.7705

Classification report:
              precision    recall  f1-score   support

          A+     0.8511    0.7339    0.7882       109
           A     0.5854    0.7500    0.6575        64
           B     0.9062    0.8286    0.8657        35

    accuracy                         0.7548       208
   macro avg     0.7809    0.7708    0.7705       208
weighted avg     0.7786    0.7548    0.7610       208


Confusion matrix (rows = actual; columns = predicted)
Class order: A+, A, B
[[80 29  0]
 [13 48  3]
 [ 1  5 29]]


In [58]:

from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_validate

# Define the three candidates.
# Each pipeline fits its own preprocessing within each CV fold.

morphology_cv_model = make_numeric_baseline()

full_combined_cv_model = clone(combined_model)

reduced_combined_cv_model = clone(reduced_combined_model)

cv_models = {
    "Morphology only": (
        morphology_cv_model,
        morphology_features,
    ),
    "All numeric + clinical": (
        full_combined_cv_model,
        combined_features,
    ),
    "Reduced numeric + clinical": (
        reduced_combined_cv_model,
        reduced_features,
    ),
}

# Training set only: do not use X_test or y_test in cross-validation.
cv_splitter = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

print("Cross-validation setup")
print("Training rows:", len(train_indices))
print("Folds:", cv_splitter.n_splits)
print("Test rows reserved:", len(test_indices))
print("\nModels to compare:")

for model_name, (_, features) in cv_models.items():
    print(f"{model_name}: {len(features)} input features")


Cross-validation setup
Training rows: 828
Folds: 5
Test rows reserved: 208

Models to compare:
Morphology only: 5 input features
All numeric + clinical: 26 input features
Reduced numeric + clinical: 24 input features


In [59]:

cv_results = []

for model_name, (model, features) in cv_models.items():
    X_train_cv = labelled_df.loc[train_indices, features]

    scores = cross_validate(
        model,
        X_train_cv,
        y_train,
        cv=cv_splitter,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "macro_f1": "f1_macro",
        },
        return_train_score=False,
        n_jobs=-1,
        error_score="raise",
    )

    result = {
        "Model": model_name,
        "Accuracy mean": scores["test_accuracy"].mean(),
        "Accuracy SD": scores["test_accuracy"].std(ddof=1),
        "Balanced accuracy mean": scores["test_balanced_accuracy"].mean(),
        "Balanced accuracy SD": scores["test_balanced_accuracy"].std(ddof=1),
        "Macro F1 mean": scores["test_macro_f1"].mean(),
        "Macro F1 SD": scores["test_macro_f1"].std(ddof=1),
    }

    cv_results.append(result)

    print(f"\n{'=' * 60}")
    print(model_name)
    print("=" * 60)

    for metric in ["accuracy", "balanced_accuracy", "macro_f1"]:
        fold_scores = scores[f"test_{metric}"]
        print(
            f"{metric}: "
            f"mean={fold_scores.mean():.4f}, "
            f"SD={fold_scores.std(ddof=1):.4f}"
        )

cv_results_df = pd.DataFrame(cv_results)

print("\nFINAL CROSS-VALIDATION COMPARISON")
print(
    cv_results_df.round(4).to_string(index=False)
)



Morphology only
accuracy: mean=0.6969, SD=0.0363
balanced_accuracy: mean=0.7196, SD=0.0368
macro_f1: mean=0.7065, SD=0.0286

All numeric + clinical
accuracy: mean=0.7271, SD=0.0151
balanced_accuracy: mean=0.7283, SD=0.0184
macro_f1: mean=0.7150, SD=0.0151

Reduced numeric + clinical
accuracy: mean=0.7295, SD=0.0160
balanced_accuracy: mean=0.7315, SD=0.0080
macro_f1: mean=0.7193, SD=0.0156

FINAL CROSS-VALIDATION COMPARISON
                     Model  Accuracy mean  Accuracy SD  Balanced accuracy mean  Balanced accuracy SD  Macro F1 mean  Macro F1 SD
           Morphology only         0.6969       0.0363                  0.7196                0.0368         0.7065       0.0286
    All numeric + clinical         0.7271       0.0151                  0.7283                0.0184         0.7150       0.0151
Reduced numeric + clinical         0.7295       0.0160                  0.7315                0.0080         0.7193       0.0156


In [60]:

# Final holdout evaluation for the provisionally selected model.
# Do not use these results to tune the model further.

final_model = clone(reduced_combined_model)

final_model.fit(X_train_reduced, y_train)
final_test_predictions = final_model.predict(X_test_reduced)

final_test_results = {
    "accuracy": accuracy_score(y_test, final_test_predictions),
    "balanced_accuracy": balanced_accuracy_score(
        y_test, final_test_predictions
    ),
    "macro_f1": f1_score(
        y_test, final_test_predictions, average="macro"
    ),
}

print("SELECTED MODEL: Reduced numeric + clinical")
print("Evaluation: Reserved holdout test set")
print("Training rows:", len(X_train_reduced))
print("Test rows:", len(X_test_reduced))

for metric, value in final_test_results.items():
    print(f"{metric}: {value:.4f}")

print("\nClassification report:")
print(
    classification_report(
        y_test,
        final_test_predictions,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

print("\nConfusion matrix (rows = actual; columns = predicted)")
print("Class order: A+, A, B")
print(
    confusion_matrix(
        y_test,
        final_test_predictions,
        labels=["A+", "A", "B"],
    )
)


SELECTED MODEL: Reduced numeric + clinical
Evaluation: Reserved holdout test set
Training rows: 828
Test rows: 208
accuracy: 0.7548
balanced_accuracy: 0.7708
macro_f1: 0.7705

Classification report:
              precision    recall  f1-score   support

          A+     0.8511    0.7339    0.7882       109
           A     0.5854    0.7500    0.6575        64
           B     0.9062    0.8286    0.8657        35

    accuracy                         0.7548       208
   macro avg     0.7809    0.7708    0.7705       208
weighted avg     0.7786    0.7548    0.7610       208


Confusion matrix (rows = actual; columns = predicted)
Class order: A+, A, B
[[80 29  0]
 [13 48  3]
 [ 1  5 29]]


In [61]:

# Investigate possible grouping identifiers.
# This is an audit only; it does not change the dataset.

possible_id_columns = [
    column
    for column in labelled_df.columns
    if any(
        keyword in column.lower()
        for keyword in [
            "donor",
            "tissue_id",
            "tissue_serial",
            "image_id",
            "cornea",
        ]
    )
]

print("Potential donor/tissue/image identifier columns:")
for column in possible_id_columns:
    print(f"\n{column}")
    print("Unique non-missing values:", labelled_df[column].nunique())
    print("Missing values:", labelled_df[column].isna().sum())
    print("Duplicate non-missing rows:", labelled_df[column].duplicated().sum())

    print("Sample values:")
    print(labelled_df[column].dropna().astype(str).head(5).to_list())


Potential donor/tissue/image identifier columns:

Tissue_ID
Unique non-missing values: 1036
Missing values: 0
Duplicate non-missing rows: 0
Sample values:
['NEB-2020-09-12654 R', 'NEB-2020-09-12655 L', 'NEB-2020-09-12656 R', 'NEB-2020-09-12657 L', 'NEB-2020-09-12658 R']

Image_ID
Unique non-missing values: 999
Missing values: 37
Duplicate non-missing rows: 36
Sample values:
['NEB-2020-09-12654 R_SP', 'NEB-2020-09-12655 L_SP', 'NEB-2020-09-12656 R_SP', 'NEB-2020-09-12657 L_SP', 'NEB-2020-09-12658 R_SP']

Overall_Donor_Quality
Unique non-missing values: 24
Missing values: 762
Duplicate non-missing rows: 1011
Sample values:
['Excellent', 'Excellent', 'Excellent', 'Excellent', 'Excellent']

Donor_Age
Unique non-missing values: 57
Missing values: 2
Duplicate non-missing rows: 978
Sample values:
['57.0', '57.0', '51.0', '51.0', '55.0']

Tissue_Serial
Unique non-missing values: 1036
Missing values: 0
Duplicate non-missing rows: 0
Sample values:
['12654', '12655', '12656', '12657', '12658']

K

In [62]:

import re
import pandas as pd

# Audit the structure of the tissue IDs.
# Do not use the derived key for splitting until its meaning is verified.

id_audit = labelled_df[["Tissue_ID", "Tissue_Serial"]].copy()

id_audit["ID_Eye_Suffix"] = (
    id_audit["Tissue_ID"]
    .astype("string")
    .str.extract(r"\s+([RL])$", expand=False)
)

id_audit["ID_Base"] = (
    id_audit["Tissue_ID"]
    .astype("string")
    .str.replace(r"\s+[RL]$", "", regex=True)
)

print("Eye suffix counts:")
print(id_audit["ID_Eye_Suffix"].value_counts(dropna=False).to_string())

print("\nRepeated base IDs after removing R/L suffix:")
base_counts = id_audit["ID_Base"].value_counts()
repeated_bases = base_counts[base_counts > 1]

print("Number of repeated base IDs:", len(repeated_bases))
print(repeated_bases.head(20).to_string())

print("\nExample ID to base-ID mapping:")
print(
    id_audit[
        ["Tissue_ID", "ID_Base", "ID_Eye_Suffix", "Tissue_Serial"]
    ].head(15).to_string(index=False)
)

print("\nTissue serial duplicate count:")
print(id_audit["Tissue_Serial"].duplicated().sum())


Eye suffix counts:
ID_Eye_Suffix
R    750
L    286

Repeated base IDs after removing R/L suffix:
Number of repeated base IDs: 0
Series([], )

Example ID to base-ID mapping:
          Tissue_ID           ID_Base ID_Eye_Suffix  Tissue_Serial
NEB-2020-09-12654 R NEB-2020-09-12654             R          12654
NEB-2020-09-12655 L NEB-2020-09-12655             L          12655
NEB-2020-09-12656 R NEB-2020-09-12656             R          12656
NEB-2020-09-12657 L NEB-2020-09-12657             L          12657
NEB-2020-09-12658 R NEB-2020-09-12658             R          12658
NEB-2020-09-12659 L NEB-2020-09-12659             L          12659
NEB-2020-09-12660 L NEB-2020-09-12660             L          12660
NEB-2020-09-12661 R NEB-2020-09-12661             R          12661
NEB-2020-09-12662 L NEB-2020-09-12662             L          12662
NEB-2020-09-12663 R NEB-2020-09-12663             R          12663
NEB-2020-09-12664 R NEB-2020-09-12664             R          12664
NEB-2020-09-12665 L NEB

In [63]:

# Search column names for possible donor, accession, and pairing identifiers.

keywords = [
    "donor",
    "patient",
    "accession",
    "pair",
    "bilateral",
    "cornea",
    "tissue",
    "serial",
    "eye",
    "record",
]

matching_columns = [
    column
    for column in df.columns
    if any(keyword in column.lower() for keyword in keywords)
]

print("Potential linkage columns:")
for column in matching_columns:
    print(f"- {column}")

print("\nPotential identifiers with repeated values:")
for column in matching_columns:
    series = df[column].dropna()
    unique_count = series.nunique()

    if len(series) > unique_count:
        print(
            f"{column}: {len(series) - unique_count} "
            f"duplicate occurrences among {len(series)} non-missing values"
        )


Potential linkage columns:
- Tissue_ID
- Eye
- Tissue_Grade
- Overall_Donor_Quality
- Donor_Age
- Tissue_Serial
- Konan_Raw_CorneaID
- Donor_Sex
- Tissue_Outcome
- Recipient_Eye
- Donor_Graft_Size_mm
- VA_Operative_Eye
- VA_Fellow_Eye
- Tissue_Grade_Source
- Tissue_Grade_Clean
- Donor_Graft_Size_Clean_mm

Potential identifiers with repeated values:
Eye: 1053 duplicate occurrences among 1055 non-missing values
Tissue_Grade: 1021 duplicate occurrences among 1036 non-missing values
Overall_Donor_Quality: 251 duplicate occurrences among 275 non-missing values
Donor_Age: 993 duplicate occurrences among 1050 non-missing values
Tissue_Serial: 1 duplicate occurrences among 1055 non-missing values
Donor_Sex: 544 duplicate occurrences among 566 non-missing values
Tissue_Outcome: 12 duplicate occurrences among 52 non-missing values
Recipient_Eye: 248 duplicate occurrences among 282 non-missing values
Donor_Graft_Size_mm: 205 duplicate occurrences among 234 non-missing values
VA_Operative_Eye: 77 

In [64]:

# Inspect duplicate Tissue_Serial records in the full cleaned dataset.

duplicate_serial_mask = df["Tissue_Serial"].notna() & df["Tissue_Serial"].duplicated(
    keep=False
)

duplicate_serial_rows = df.loc[
    duplicate_serial_mask,
    [
        "Tissue_ID",
        "Eye",
        "Tissue_Serial",
        "Image_ID",
        "Konan_Raw_CorneaID",
        "Donor_Age",
        "Donor_Sex",
        TARGET,
    ],
]

print("Rows with duplicated Tissue_Serial values:")
print(duplicate_serial_rows.to_string(index=False))

print("\nNumber of rows involved:", len(duplicate_serial_rows))


Rows with duplicated Tissue_Serial values:
                   Tissue_ID   Eye  Tissue_Serial               Image_ID  Konan_Raw_CorneaID  Donor_Age Donor_Sex Tissue_Grade_Clean
14302 L (year/month unknown)  Left          14302                    NaN                 NaN        NaN       NaN                NaN
         NEB-2022-10-14302 R Right          14302 NEB-2022-10-14302 R_SP NEB-2022-10-14302 R       58.0      Male                  A

Number of rows involved: 2


In [65]:

# Inspect the incomplete record in the cleaned dataset.

mask = df["Tissue_ID"].astype("string").str.contains(
    r"14302 L \(year/month unknown\)",
    case=False,
    na=False,
)

print("Matching rows in cleaned dataset:")
print(df.loc[mask].to_string(index=False))

print("\nMatching rows in original dataframe, if available:")
if "raw_df" in globals():
    raw_mask = raw_df["Tissue_ID"].astype("string").str.contains(
        r"14302 L \(year/month unknown\)",
        case=False,
        na=False,
    )
    print(raw_df.loc[raw_mask].to_string(index=False))
else:
    print("raw_df is not available in the current notebook.")


Matching rows in cleaned dataset:
                   Tissue_ID  Eye Image_ID Disease_Label Disease_Severity  ECD_CD_cells_per_mm2  CV  HEX_percent  SD  Total_Cell_Count  Total_Area_um2 Epithelium_Findings Epithelium_Severity Stroma_Findings Stroma_Severity Descemet_Findings Descemet_Severity Endothelium_Findings Endothelium_Severity Additional_Findings Tissue_Grade Overall_Donor_Quality Transplant_Suitability Suitability_Reason Primary_COD  Donor_Age  Tissue_Serial             Linkage_Status Source_PDF  PDF_Pages Konan_Report_File SP_Image_File Cell_Image_File Konan_Raw_CorneaID Konan_Exam_Date  Konan_Mean_Cell_Area_um2 Konan_Comments Donor_Sex Admitting_Diagnosis Death_Date_Time Preservation_Date_Time Death_to_Preservation_Time Storage_Medium Serology_HBsAg Serology_HCV Serology_HIV Serology_VDRL Lens_Status  Form_ECD_cells_per_mm2 Intended_Use Tissue_Outcome Recipient_Age Recipient_Eye Recipient_Indication Request_Indication Previous_Grafts Primary_Procedure Secondary_Procedures Surg

Rows removed: 1
Rows remaining: 1054
Record still present: False


In [67]:

print("Dataset shape:", df.shape)

print(
    "Removed record present:",
    df["Tissue_ID"].eq("14302 L (year/month unknown)").any()
)

labelled_df = df.loc[df["Tissue_Grade_Clean"].isin(["A+", "A", "B"])].copy()

print("\nLabelled rows:", len(labelled_df))
print("\nGrade distribution:")
print(labelled_df["Tissue_Grade_Clean"].value_counts().to_string())


Dataset shape: (1054, 116)
Removed record present: False

Labelled rows: 1036

Grade distribution:
Tissue_Grade_Clean
A+    545
A     319
B     172


In [68]:

# Derive a candidate grouping ID by removing the final R/L eye suffix.
# This assumes the shared accession base identifies the donor.

labelled_df["Donor_Group_Candidate"] = (
    labelled_df["Tissue_ID"]
    .astype("string")
    .str.replace(r"\s+[RL]$", "", regex=True)
)

print("Labelled rows:", len(labelled_df))
print(
    "Unique candidate donor groups:",
    labelled_df["Donor_Group_Candidate"].nunique()
)

print(
    "Rows without a candidate group:",
    labelled_df["Donor_Group_Candidate"].isna().sum()
)

print("\nCandidate groups containing multiple rows:")
group_sizes = labelled_df["Donor_Group_Candidate"].value_counts()
print(group_sizes[group_sizes > 1].to_string())

print("\nSample mapping:")
print(
    labelled_df[
        ["Tissue_ID", "Donor_Group_Candidate", TARGET]
    ].head(10).to_string(index=False)
)


Labelled rows: 1036
Unique candidate donor groups: 1036
Rows without a candidate group: 0

Candidate groups containing multiple rows:
Series([], )

Sample mapping:
          Tissue_ID Donor_Group_Candidate Tissue_Grade_Clean
NEB-2020-09-12654 R     NEB-2020-09-12654                  B
NEB-2020-09-12655 L     NEB-2020-09-12655                  B
NEB-2020-09-12656 R     NEB-2020-09-12656                  A
NEB-2020-09-12657 L     NEB-2020-09-12657                 A+
NEB-2020-09-12658 R     NEB-2020-09-12658                 A+
NEB-2020-09-12659 L     NEB-2020-09-12659                 A+
NEB-2020-09-12660 L     NEB-2020-09-12660                 A+
NEB-2020-09-12661 R     NEB-2020-09-12661                 A+
NEB-2020-09-12662 L     NEB-2020-09-12662                 A+
NEB-2020-09-12663 R     NEB-2020-09-12663                 A+


In [69]:

# Check whether candidate accession bases repeat in the full dataset.

df["Donor_Group_Candidate"] = (
    df["Tissue_ID"]
    .astype("string")
    .str.replace(r"\s+[RL]$", "", regex=True)
)

group_counts = df["Donor_Group_Candidate"].value_counts()

repeated_groups = group_counts[group_counts > 1]

print("Full dataset rows:", len(df))
print("Unique candidate groups:", df["Donor_Group_Candidate"].nunique())
print("Repeated candidate groups:", len(repeated_groups))

if len(repeated_groups) > 0:
    print("\nRepeated candidate groups:")
    for group_id in repeated_groups.index:
        print(f"\nGroup: {group_id}")
        print(
            df.loc[
                df["Donor_Group_Candidate"] == group_id,
                ["Tissue_ID", "Eye", "Tissue_Grade_Clean", "Image_ID"],
            ].to_string(index=False)
        )
else:
    print("No repeated candidate groups found.")


Full dataset rows: 1054
Unique candidate groups: 1054
Repeated candidate groups: 0
No repeated candidate groups found.


In [70]:

# Identify correctly and incorrectly classified test cases.

error_analysis_df = labelled_df.loc[
    test_indices,
    [
        "Tissue_ID",
        "Tissue_Grade_Clean",
        "ECD_CD_cells_per_mm2",
        "CV",
        "HEX_percent",
        "SD",
        "Total_Cell_Count",
        "Total_Area_um2",
        "Disease_Label",
        "Disease_Severity",
    ],
].copy()

error_analysis_df["Predicted_Grade"] = final_test_predictions
error_analysis_df["Correct"] = (
    error_analysis_df["Tissue_Grade_Clean"]
    == error_analysis_df["Predicted_Grade"]
)

print("Test cases:", len(error_analysis_df))
print("Correct:", int(error_analysis_df["Correct"].sum()))
print("Incorrect:", int((~error_analysis_df["Correct"]).sum()))
print(
    "Accuracy:",
    round(error_analysis_df["Correct"].mean(), 4),
)

print("\nErrors by actual grade:")
print(
    error_analysis_df.loc[~error_analysis_df["Correct"]]
    ["Tissue_Grade_Clean"]
    .value_counts()
    .reindex(["A+", "A", "B"], fill_value=0)
    .to_string()
)

print("\nMisclassified cases:")
print(
    error_analysis_df.loc[
        ~error_analysis_df["Correct"]
    ].to_string(index=False)
)


Test cases: 208
Correct: 157
Incorrect: 51
Accuracy: 0.7548

Errors by actual grade:
Tissue_Grade_Clean
A+    29
A     16
B      6

Misclassified cases:
          Tissue_ID Tissue_Grade_Clean  ECD_CD_cells_per_mm2   CV  HEX_percent    SD  Total_Cell_Count  Total_Area_um2                                         Disease_Label Disease_Severity Predicted_Grade  Correct
NEB-2023-11-15526 R                  B                2247.0 42.0         41.0 187.0             102.0         45387.0 FECD; Polymorphism and pleomorphism; Low cell density             Mild               A    False
NEB-2020-09-12665 L                  A                2710.0 33.0         70.0 121.0             112.0         41299.0                                                 Folds             Mild              A+    False
NEB-2023-12-15605 R                  A                2857.0 38.0         45.0 134.0             111.0         38851.0                                                Normal              NaN             

In [71]:
# Summarise actual grade versus predicted grade.
print("ACTUAL GRADE vs PREDICTED GRADE")
print(
    pd.crosstab(
        error_analysis_df["Tissue_Grade_Clean"],
        error_analysis_df["Predicted_Grade"],
    )
    .reindex(index=["A+", "A", "B"], columns=["A+", "A", "B"], fill_value=0)
    .to_string()
)

# Check whether missing numeric measurements are associated with errors.
measurement_columns = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
]

print("\nMISSING MEASUREMENTS: CORRECT vs INCORRECT PREDICTIONS")

missing_summary = error_analysis_df.groupby("Correct")[measurement_columns].apply(
    lambda group: group.isna().any(axis=1).mean()
)

print(
    missing_summary.rename(index={True: "Correct", False: "Incorrect"})
    .rename("Fraction with at least one missing measurement")
    .round(4)
    .to_string()
)

print("\nERRORS BY ACTUAL GRADE AND PREDICTED GRADE")
errors_only = error_analysis_df.loc[~error_analysis_df["Correct"]]

print(
    pd.crosstab(
        errors_only["Tissue_Grade_Clean"],
        errors_only["Predicted_Grade"],
    )
    .reindex(index=["A+", "A", "B"], columns=["A+", "A", "B"], fill_value=0)
    .to_string()
)

ACTUAL GRADE vs PREDICTED GRADE
Predicted_Grade     A+   A   B
Tissue_Grade_Clean            
A+                  80  29   0
A                   13  48   3
B                    1   5  29

MISSING MEASUREMENTS: CORRECT vs INCORRECT PREDICTIONS
Correct
Incorrect    0.1765
Correct      0.0573

ERRORS BY ACTUAL GRADE AND PREDICTED GRADE
Predicted_Grade     A+   A  B
Tissue_Grade_Clean           
A+                   0  29  0
A                   13   0  3
B                    1   5  0


In [72]:
# Compare correctly and incorrectly predicted A+ cases.
a_plus_cases = error_analysis_df[
    error_analysis_df["Tissue_Grade_Clean"] == "A+"
].copy()

a_plus_cases["Prediction_Status"] = a_plus_cases["Correct"].map({
    True: "Correctly predicted A+",
    False: "Misclassified as another grade",
})

comparison_columns = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
]

print("A+ CASES: CORRECT vs INCORRECT PREDICTIONS")
print(
    a_plus_cases.groupby("Prediction_Status")[comparison_columns]
    .agg(["count", "median", "mean"])
    .round(2)
    .to_string()
)

print("\nNUMBER OF CASES IN EACH GROUP")
print(a_plus_cases["Prediction_Status"].value_counts().to_string())

A+ CASES: CORRECT vs INCORRECT PREDICTIONS
                               ECD_CD_cells_per_mm2                     CV               HEX_percent                  SD                Total_Cell_Count                Total_Area_um2                   
                                              count  median     mean count median   mean       count median   mean count median    mean            count median    mean          count   median      mean
Prediction_Status                                                                                                                                                                                        
Correctly predicted A+                           76  2874.0  2874.01    76   36.0  36.45          76   55.5  55.93    76  125.0  127.29               76  107.0  107.89             76  38405.5  37826.33
Misclassified as another grade                   24  2535.0  2579.75    24   39.0  39.00          24   51.0  51.33    24  152.0  151.54              

In [73]:
# Examine ECD-based grades for actual A+ test cases.
a_plus_cases = error_analysis_df[
    error_analysis_df["Tissue_Grade_Clean"] == "A+"
].copy()

def ecd_rule_grade(ecd):
    if pd.isna(ecd):
        return "Missing ECD"
    if ecd > 3000:
        return "A+"
    if ecd >= 2500:
        return "A"
    if ecd >= 2000:
        return "B"
    return "Below B threshold"

a_plus_cases["ECD_Rule_Grade"] = (
    a_plus_cases["ECD_CD_cells_per_mm2"].apply(ecd_rule_grade)
)

print("ECD RULE FOR ACTUAL A+ CASES")
print(
    pd.crosstab(
        a_plus_cases["Prediction_Status"]
        if "Prediction_Status" in a_plus_cases.columns
        else a_plus_cases["Correct"].map({
            True: "Correct model prediction",
            False: "Incorrect model prediction",
        }),
        a_plus_cases["ECD_Rule_Grade"],
    ).to_string()
)

print("\nECD RULE DISTRIBUTION AMONG MISCLASSIFIED A+ CASES")
print(
    a_plus_cases.loc[~a_plus_cases["Correct"], "ECD_Rule_Grade"]
    .value_counts()
    .to_string()
)

ECD RULE FOR ACTUAL A+ CASES
ECD_Rule_Grade               A  A+  B  Missing ECD
Correct                                           
Correct model prediction    54  22  0            4
Incorrect model prediction  23   0  1            5

ECD RULE DISTRIBUTION AMONG MISCLASSIFIED A+ CASES
ECD_Rule_Grade
A              23
Missing ECD     5
B               1


In [74]:
# Compare expert-assigned grades with the ECD-only rule
all_test_cases = error_analysis_df.copy()

all_test_cases["ECD_Rule_Grade"] = (
    all_test_cases["ECD_CD_cells_per_mm2"].apply(ecd_rule_grade)
)

print("ECD RULE vs EXPERT GRADE")
print(
    pd.crosstab(
        all_test_cases["Tissue_Grade_Clean"],
        all_test_cases["ECD_Rule_Grade"],
    )
    .reindex(
        index=["A+", "A", "B"],
        columns=["A+", "A", "B", "Missing ECD", "Below B threshold"],
        fill_value=0,
    )
    .to_string()
)

print("\nECD RULE AGREEMENT WITH EXPERT GRADE")
evaluable = all_test_cases[
    all_test_cases["ECD_Rule_Grade"].isin(["A+", "A", "B"])
]

print("Cases with evaluable ECD:", len(evaluable))
print(
    "Agreement among evaluable cases:",
    round(
        (
            evaluable["Tissue_Grade_Clean"]
            == evaluable["ECD_Rule_Grade"]
        ).mean(),
        4,
    ),
)
print("Cases with missing or below-threshold ECD:", len(all_test_cases) - len(evaluable))

ECD RULE vs EXPERT GRADE
ECD_Rule_Grade      A+   A   B  Missing ECD  Below B threshold
Tissue_Grade_Clean                                            
A+                  22  77   1            9                  0
A                    3  20  36            5                  0
B                    1   0  17            4                 13

ECD RULE AGREEMENT WITH EXPERT GRADE
Cases with evaluable ECD: 177
Agreement among evaluable cases: 0.3333
Cases with missing or below-threshold ECD: 31


In [75]:
# Compare the model and ECD rule on identical test cases.
comparison_df = error_analysis_df.copy()
comparison_df["ECD_Rule_Grade"] = (
    comparison_df["ECD_CD_cells_per_mm2"].apply(ecd_rule_grade)
)

evaluable_df = comparison_df[
    comparison_df["ECD_Rule_Grade"].isin(["A+", "A", "B"])
].copy()

y_true = evaluable_df["Tissue_Grade_Clean"]
y_model = evaluable_df["Predicted_Grade"]
y_ecd = evaluable_df["ECD_Rule_Grade"]

print("Matched evaluation cases:", len(evaluable_df))

for name, predictions in [
    ("Combined model", y_model),
    ("ECD-only rule", y_ecd),
]:
    print(f"\n{name}")
    print("Accuracy:", round(accuracy_score(y_true, predictions), 4))
    print(
        "Balanced accuracy:",
        round(balanced_accuracy_score(y_true, predictions), 4),
    )
    print(
        "Macro F1:",
        round(
            f1_score(y_true, predictions, labels=["A+", "A", "B"], average="macro"),
            4,
        ),
    )


Matched evaluation cases: 177

Combined model
Accuracy: 0.7627
Balanced accuracy: 0.7797
Macro F1: 0.7729

ECD-only rule
Accuracy: 0.3333
Balanced accuracy: 0.5011
Macro F1: 0.3593


In [76]:
from sklearn.metrics import classification_report

print("COMBINED MODEL — PER-CLASS PERFORMANCE")
print(
    classification_report(
        y_true,
        y_model,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

print("\nECD-ONLY RULE — PER-CLASS PERFORMANCE")
print(
    classification_report(
        y_true,
        y_ecd,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    )
)

COMBINED MODEL — PER-CLASS PERFORMANCE
              precision    recall  f1-score   support

          A+     0.8636    0.7600    0.8085       100
           A     0.6197    0.7458    0.6769        59
           B     0.8333    0.8333    0.8333        18

    accuracy                         0.7627       177
   macro avg     0.7722    0.7797    0.7729       177
weighted avg     0.7792    0.7627    0.7672       177


ECD-ONLY RULE — PER-CLASS PERFORMANCE
              precision    recall  f1-score   support

          A+     0.8462    0.2200    0.3492       100
           A     0.2062    0.3390    0.2564        59
           B     0.3148    0.9444    0.4722        18

    accuracy                         0.3333       177
   macro avg     0.4557    0.5011    0.3593       177
weighted avg     0.5788    0.3333    0.3308       177



In [77]:
print("Labelled dataset:", labelled_df.shape)
print("Target column:", TARGET)

print("\nAvailable model objects:")
for name in [
    "numeric_model",
    "clinical_model",
    "combined_model",
    "reduced_combined_model",
    "final_model",
]:
    print(f"{name}: {name in globals()}")

print("\nCross-validation results available:", "cv_results_df" in globals())
print("Final test predictions available:", "final_test_predictions" in globals())
print("Matched comparison dataset available:", "evaluable_df" in globals())

Labelled dataset: (1036, 117)
Target column: Tissue_Grade_Clean

Available model objects:
numeric_model: False
clinical_model: False
combined_model: True
reduced_combined_model: True
final_model: True

Cross-validation results available: True
Final test predictions available: True
Matched comparison dataset available: True


In [78]:
# Verify the current modelling and evaluation setup.

print("DATASET")
print("Labelled rows:", len(labelled_df))
print("Target:", TARGET)
print("Target distribution:")
print(labelled_df[TARGET].value_counts().reindex(["A+", "A", "B"]).to_string())

print("\nTRAIN / TEST SPLIT")
print("Training rows:", len(train_indices))
print("Test rows:", len(test_indices))
print("Training and test indices overlap:",
      len(set(train_indices).intersection(set(test_indices))))

print("\nMODEL")
print("Selected model:", type(final_model).__name__)
print("Input features:", len(reduced_features))
print("Final test predictions:", len(final_test_predictions))

print("\nCROSS-VALIDATION")
print(cv_results_df[
    ["Model", "Macro F1 mean", "Macro F1 SD"]
].round(4).to_string(index=False))

DATASET
Labelled rows: 1036
Target: Tissue_Grade_Clean
Target distribution:
Tissue_Grade_Clean
A+    545
A     319
B     172

TRAIN / TEST SPLIT
Training rows: 828
Test rows: 208
Training and test indices overlap: 0

MODEL
Selected model: Pipeline
Input features: 24
Final test predictions: 208

CROSS-VALIDATION
                     Model  Macro F1 mean  Macro F1 SD
           Morphology only         0.7065       0.0286
    All numeric + clinical         0.7150       0.0151
Reduced numeric + clinical         0.7193       0.0156


In [79]:
print("SELECTED MODEL PIPELINE")
print(final_model)

print("\nSELECTED INPUT FEATURES")
for i, feature in enumerate(reduced_features, start=1):
    print(f"{i:02d}. {feature}")

print("\nCROSS-VALIDATION SUMMARY")
print(cv_results_df.to_string(index=False))

SELECTED MODEL PIPELINE
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['ECD_CD_cells_per_mm2', 'CV',
                                                   'HEX_percent', 'SD',
                                                   'Total_Cell_Count',
                                                   'Total_Area_um2']),
                                                 ('clinical_binary',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(fill_value=-1,

In [81]:
print("X_train type:", type(X_train))
print("X_train shape:", X_train.shape)

print("\nFirst 30 X_train columns:")
print(X_train.columns[:30].tolist())

print("\nRequired model features missing from X_train:")
print([feature for feature in reduced_features
       if feature not in X_train.columns])

print("\nRequired model features available in X_train:")
print([feature for feature in reduced_features
       if feature in X_train.columns])

X_train type: <class 'pandas.DataFrame'>
X_train shape: (828, 5)

First 30 X_train columns:
['CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2']

Required model features missing from X_train:
['ECD_CD_cells_per_mm2', 'Stroma_Is_Clear', 'Stroma_Is_Compact', 'Stroma_Has_Arcus', 'Stroma_Has_Edema', 'Stroma_Has_Scar', 'Epi_Has_Arcus', 'Epi_Has_Defect', 'Epi_Has_Exposure', 'Epi_Has_Haze', 'Endo_Has_Stress_Lines', 'Endo_Has_RBC', 'Endo_Has_Cell_Dropout', 'Descemet_Has_Folds', 'Descemet_Fold_Severity', 'Descemet_Fold_Relative_Number', 'Endo_Stress_Line_Location', 'Stroma_Arcus_Severity', 'Epi_Exposure_Severity']

Required model features available in X_train:
['CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2']


In [82]:
print("AVAILABLE DATAFRAME VARIABLES")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        cols = list(obj.columns)

        has_ecd = "ECD_CD_cells_per_mm2" in cols
        matched_features = sum(
            feature in cols for feature in reduced_features
        )

        if matched_features >= 6 or has_ecd:
            print(
                f"{name}: shape={obj.shape}, "
                f"matched model features={matched_features}/"
                f"{len(reduced_features)}, "
                f"has ECD={has_ecd}"
            )

AVAILABLE DATAFRAME VARIABLES
df: shape=(1054, 117), matched model features=24/24, has ECD=True
_1: shape=(5, 116), matched model features=24/24, has ECD=True
labelled_df: shape=(1036, 117), matched model features=24/24, has ECD=True
duplicate_serials: shape=(0, 116), matched model features=24/24, has ECD=True
X_clinical: shape=(1036, 20), matched model features=18/24, has ECD=False
X_combined: shape=(1036, 26), matched model features=24/24, has ECD=True
X_train_ecd: shape=(828, 1), matched model features=1/24, has ECD=True
X_test_ecd: shape=(208, 1), matched model features=1/24, has ECD=True
X_train_numeric: shape=(828, 6), matched model features=6/24, has ECD=True
X_test_numeric: shape=(208, 6), matched model features=6/24, has ECD=True
X_train_clinical: shape=(828, 20), matched model features=18/24, has ECD=False
X_test_clinical: shape=(208, 20), matched model features=18/24, has ECD=False
X_train_combined: shape=(828, 26), matched model features=24/24, has ECD=True
X_test_combined:

In [83]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate, StratifiedKFold
from sklearn.pipeline import Pipeline

# Reuse preprocessing from the existing reduced model.
rf_preprocessor = clone(final_model.named_steps["preprocessor"])

random_forest_model = Pipeline(steps=[
    ("preprocessor", rf_preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf_results = cross_validate(
    random_forest_model,
    X_train_reduced,
    y_train,
    cv=rf_cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "macro_f1": "f1_macro"
    },
    n_jobs=-1,
    return_train_score=False
)

print("RANDOM FOREST — 5-FOLD TRAINING CROSS-VALIDATION")
print(f"Accuracy:          {rf_results['test_accuracy'].mean():.4f} "
      f"+/- {rf_results['test_accuracy'].std():.4f}")
print(f"Balanced accuracy: {rf_results['test_balanced_accuracy'].mean():.4f} "
      f"+/- {rf_results['test_balanced_accuracy'].std():.4f}")
print(f"Macro F1:          {rf_results['test_macro_f1'].mean():.4f} "
      f"+/- {rf_results['test_macro_f1'].std():.4f}")

print("\nEXISTING REDUCED LOGISTIC REGRESSION")
print("Accuracy: 0.7295 +/- 0.0160")
print("Balanced accuracy: 0.7315 +/- 0.0080")
print("Macro F1: 0.7193 +/- 0.0156")

RANDOM FOREST — 5-FOLD TRAINING CROSS-VALIDATION
Accuracy:          0.8382 +/- 0.0178
Balanced accuracy: 0.8267 +/- 0.0325
Macro F1:          0.8337 +/- 0.0262

EXISTING REDUCED LOGISTIC REGRESSION
Accuracy: 0.7295 +/- 0.0160
Balanced accuracy: 0.7315 +/- 0.0080
Macro F1: 0.7193 +/- 0.0156


In [84]:
print("TRAINING DATA CHECK")
print("X_train_reduced:", X_train_reduced.shape)
print("y_train:", y_train.shape)
print("Random Forest CV splits:", rf_cv.get_n_splits())

print("\nRANDOM FOREST FOLD MACRO F1")
for i, score in enumerate(rf_results["test_macro_f1"], start=1):
    print(f"Fold {i}: {score:.4f}")

print("\nEXISTING LOGISTIC REGRESSION CV RESULTS")
print(cv_results_df.to_string(index=False))

TRAINING DATA CHECK
X_train_reduced: (828, 24)
y_train: (828,)
Random Forest CV splits: 5

RANDOM FOREST FOLD MACRO F1
Fold 1: 0.8383
Fold 2: 0.8572
Fold 3: 0.7835
Fold 4: 0.8516
Fold 5: 0.8381

EXISTING LOGISTIC REGRESSION CV RESULTS
                     Model  Accuracy mean  Accuracy SD  Balanced accuracy mean  Balanced accuracy SD  Macro F1 mean  Macro F1 SD
           Morphology only       0.696860     0.036277                0.719610              0.036774       0.706486     0.028649
    All numeric + clinical       0.727076     0.015104                0.728341              0.018427       0.715002     0.015072
Reduced numeric + clinical       0.729478     0.015975                0.731472              0.008000       0.719251     0.015596


In [85]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

# Out-of-fold predictions using the same five-fold CV configuration.
rf_oof_predictions = cross_val_predict(
    random_forest_model,
    X_train_reduced,
    y_train,
    cv=rf_cv,
    n_jobs=-1,
    method="predict"
)

print("RANDOM FOREST — OUT-OF-FOLD CLASSIFICATION REPORT")
print(classification_report(
    y_train,
    rf_oof_predictions,
    labels=["A+", "A", "B"],
    zero_division=0,
    digits=4
))

print("RANDOM FOREST — OUT-OF-FOLD CONFUSION MATRIX")
print("Rows = actual grade; columns = predicted grade")
print("Class order: A+, A, B")
print(confusion_matrix(
    y_train,
    rf_oof_predictions,
    labels=["A+", "A", "B"]
))

RANDOM FOREST — OUT-OF-FOLD CLASSIFICATION REPORT
              precision    recall  f1-score   support

          A+     0.8802    0.8761    0.8782       436
           A     0.7343    0.7804    0.7567       255
           B     0.9187    0.8248    0.8692       137

    accuracy                         0.8382       828
   macro avg     0.8444    0.8271    0.8347       828
weighted avg     0.8416    0.8382    0.8393       828

RANDOM FOREST — OUT-OF-FOLD CONFUSION MATRIX
Rows = actual grade; columns = predicted grade
Class order: A+, A, B
[[382  52   2]
 [ 48 199   8]
 [  4  20 113]]


In [86]:
import pandas as pd
import numpy as np

from sklearn.base import clone
from sklearn.inspection import permutation_importance
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

# Calculate permutation importance separately on each validation fold.
importance_records = []

for fold, (train_idx, val_idx) in enumerate(
    rf_cv.split(X_train_reduced, y_train), start=1
):
    X_fold_train = X_train_reduced.iloc[train_idx]
    X_fold_val = X_train_reduced.iloc[val_idx]

    y_fold_train = y_train.iloc[train_idx]
    y_fold_val = y_train.iloc[val_idx]

    fold_model = clone(random_forest_model)
    fold_model.fit(X_fold_train, y_fold_train)

    fold_importance = permutation_importance(
        fold_model,
        X_fold_val,
        y_fold_val,
        scoring="f1_macro",
        n_repeats=10,
        random_state=42,
        n_jobs=-1
    )

    for feature, mean_imp, std_imp in zip(
        X_train_reduced.columns,
        fold_importance.importances_mean,
        fold_importance.importances_std
    ):
        importance_records.append({
            "Fold": fold,
            "Feature": feature,
            "Importance": mean_imp,
            "Within_fold_SD": std_imp
        })

importance_df = pd.DataFrame(importance_records)

importance_summary = (
    importance_df
    .groupby("Feature")
    .agg(
        Mean_Importance=("Importance", "mean"),
        Across_Fold_SD=("Importance", "std")
    )
    .sort_values("Mean_Importance", ascending=False)
)

print("TOP 15 FEATURES — CROSS-VALIDATED PERMUTATION IMPORTANCE")
print(importance_summary.head(15).round(4).to_string())

print("\nBOTTOM 5 FEATURES")
print(importance_summary.tail(5).round(4).to_string())

print("\nFeatures with negative mean importance:",
      int((importance_summary["Mean_Importance"] < 0).sum()))

TOP 15 FEATURES — CROSS-VALIDATED PERMUTATION IMPORTANCE
                           Mean_Importance  Across_Fold_SD
Feature                                                   
ECD_CD_cells_per_mm2                0.3811          0.0407
HEX_percent                         0.0314          0.0222
Total_Area_um2                      0.0087          0.0065
SD                                  0.0053          0.0038
Stroma_Has_Scar                     0.0047          0.0069
Endo_Has_RBC                        0.0046          0.0076
Epi_Has_Defect                      0.0001          0.0045
Stroma_Is_Clear                    -0.0006          0.0034
Descemet_Fold_Severity             -0.0011          0.0097
Endo_Has_Stress_Lines              -0.0013          0.0076
Epi_Has_Haze                       -0.0024          0.0093
Stroma_Is_Compact                  -0.0025          0.0048
Endo_Stress_Line_Location          -0.0028          0.0040
Stroma_Has_Arcus                   -0.0029          0.0076

In [87]:
# Feature ablation groups for Random Forest.
# All groups use the same training rows and cross-validation folds.

ecd_features = [
    "ECD_CD_cells_per_mm2"
]

morphology_without_ecd = [
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
]

clinical_features = [
    "Stroma_Is_Clear",
    "Stroma_Is_Compact",
    "Stroma_Has_Arcus",
    "Stroma_Has_Edema",
    "Stroma_Has_Scar",
    "Epi_Has_Arcus",
    "Epi_Has_Defect",
    "Epi_Has_Exposure",
    "Epi_Has_Haze",
    "Endo_Has_Stress_Lines",
    "Endo_Has_RBC",
    "Endo_Has_Cell_Dropout",
    "Descemet_Has_Folds",
    "Descemet_Fold_Severity",
    "Descemet_Fold_Relative_Number",
    "Endo_Stress_Line_Location",
    "Stroma_Arcus_Severity",
    "Epi_Exposure_Severity",
]

feature_groups = {
    "ECD only": ecd_features,
    "Morphology without ECD": morphology_without_ecd,
    "Clinical only": clinical_features,
    "Morphology + clinical (no ECD)": (
        morphology_without_ecd + clinical_features
    ),
    "All features": reduced_features,
}

print("FEATURE GROUP VALIDATION")
print("Training rows:", len(X_train_reduced))
print("Target rows:", len(y_train))

for group_name, features in feature_groups.items():
    missing = [
        feature for feature in features
        if feature not in X_train_reduced.columns
    ]

    print(f"\n{group_name}")
    print("Feature count:", len(features))
    print("Missing columns:", missing if missing else "None")

FEATURE GROUP VALIDATION
Training rows: 828
Target rows: 828

ECD only
Feature count: 1
Missing columns: None

Morphology without ECD
Feature count: 5
Missing columns: None

Clinical only
Feature count: 18
Missing columns: None

Morphology + clinical (no ECD)
Feature count: 23
Missing columns: None

All features
Feature count: 24
Missing columns: None


In [89]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

ablation_results = []

for group_name, features in feature_groups.items():
    X_group = X_train_reduced[features].copy()

    numeric_features = [
        f for f in features
        if f in ecd_features + morphology_without_ecd
    ]
    binary_features = [
        f for f in features
        if f in clinical_binary_candidates
    ]
    categorical_features = [
        f for f in features
        if f in clinical_categorical_candidates
    ]

    transformers = []

    if numeric_features:
        transformers.append((
            "numeric",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ))

    if binary_features:
        transformers.append((
            "clinical_binary",
            Pipeline(steps=[
                ("imputer", SimpleImputer(
                    strategy="constant", fill_value=-1
                )),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            binary_features
        ))

    if categorical_features:
        transformers.append((
            "clinical_categorical",
            Pipeline(steps=[
                ("imputer", SimpleImputer(
                    strategy="constant", fill_value="__MISSING__"
                )),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ))

    group_preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

    group_model = Pipeline(steps=[
        ("preprocessor", group_preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    scores = cross_validate(
        group_model,
        X_group,
        y_train,
        cv=rf_cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "macro_f1": "f1_macro"
        },
        n_jobs=-1,
        return_train_score=False
    )

    ablation_results.append({
        "Feature group": group_name,
        "Features": len(features),
        "Accuracy mean": scores["test_accuracy"].mean(),
        "Balanced accuracy mean": scores["test_balanced_accuracy"].mean(),
        "Macro F1 mean": scores["test_macro_f1"].mean(),
        "Macro F1 SD": scores["test_macro_f1"].std()
    })

ablation_results_df = (
    pd.DataFrame(ablation_results)
    .sort_values("Macro F1 mean", ascending=False)
    .reset_index(drop=True)
)

print("RANDOM FOREST — FEATURE ABLATION RESULTS")
print(ablation_results_df.round(4).to_string(index=False))

RANDOM FOREST — FEATURE ABLATION RESULTS
                 Feature group  Features  Accuracy mean  Balanced accuracy mean  Macro F1 mean  Macro F1 SD
                  All features        24         0.8382                  0.8267         0.8337       0.0262
                      ECD only         1         0.7717                  0.7722         0.7722       0.0270
        Morphology without ECD         5         0.7778                  0.7609         0.7669       0.0421
Morphology + clinical (no ECD)        23         0.7561                  0.7377         0.7400       0.0577
                 Clinical only        18         0.3369                  0.3716         0.3147       0.0502


In [90]:
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import cross_validate

comparison_groups = {
    "ECD + morphology": ecd_features + morphology_without_ecd,
    "ECD + morphology + clinical": reduced_features,
}

fold_results = {}

for group_name, features in comparison_groups.items():
    # Reuse the correctly configured preprocessing from the ablation experiment.
    numeric_features = [
        f for f in features
        if f in ecd_features + morphology_without_ecd
    ]
    binary_features = [
        f for f in features
        if f in clinical_binary_candidates
    ]
    categorical_features = [
        f for f in features
        if f in clinical_categorical_candidates
    ]

    transformers = []

    if numeric_features:
        transformers.append((
            "numeric",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_features
        ))

    if binary_features:
        transformers.append((
            "clinical_binary",
            Pipeline(steps=[
                ("imputer", SimpleImputer(
                    strategy="constant", fill_value=-1
                )),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            binary_features
        ))

    if categorical_features:
        transformers.append((
            "clinical_categorical",
            Pipeline(steps=[
                ("imputer", SimpleImputer(
                    strategy="constant", fill_value="__MISSING__"
                )),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        ))

    comparison_model = Pipeline(steps=[
        ("preprocessor", ColumnTransformer(
            transformers=transformers,
            remainder="drop"
        )),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    X_group = X_train_reduced[features].copy()

    scores = cross_validate(
        comparison_model,
        X_group,
        y_train,
        cv=rf_cv,
        scoring="f1_macro",
        n_jobs=-1,
        return_train_score=False
    )

    fold_results[group_name] = scores["test_score"]

print("PER-FOLD MACRO F1 COMPARISON")
print("Fold | ECD + morphology | ECD + morphology + clinical | Difference")

for i, (numeric_score, combined_score) in enumerate(
    zip(
        fold_results["ECD + morphology"],
        fold_results["ECD + morphology + clinical"]
    ),
    start=1
):
    print(
        f"{i:>4} | {numeric_score:.4f}"
        f"           | {combined_score:.4f}"
        f"                        | {combined_score - numeric_score:+.4f}"
    )

differences = (
    fold_results["ECD + morphology + clinical"]
    - fold_results["ECD + morphology"]
)

print("\nSUMMARY")
print(f"Numeric-only mean macro F1: "
      f"{fold_results['ECD + morphology'].mean():.4f}")
print(f"Combined mean macro F1:     "
      f"{fold_results['ECD + morphology + clinical'].mean():.4f}")
print(f"Mean paired difference:     {differences.mean():+.4f}")
print(f"Folds improved by adding clinical features: "
      f"{(differences > 0).sum()} / {len(differences)}")

PER-FOLD MACRO F1 COMPARISON
Fold | ECD + morphology | ECD + morphology + clinical | Difference
   1 | 0.8554           | 0.8383                        | -0.0171
   2 | 0.8386           | 0.8572                        | +0.0186
   3 | 0.7744           | 0.7835                        | +0.0091
   4 | 0.8531           | 0.8516                        | -0.0016
   5 | 0.8442           | 0.8381                        | -0.0062

SUMMARY
Numeric-only mean macro F1: 0.8332
Combined mean macro F1:     0.8337
Mean paired difference:     +0.0006
Folds improved by adding clinical features: 2 / 5


In [91]:
morphology_feature_groups = {
    "ECD only": ecd_features,
    "ECD + CV": ecd_features + ["CV"],
    "ECD + HEX": ecd_features + ["HEX_percent"],
    "ECD + SD": ecd_features + ["SD"],
    "ECD + total cell count": ecd_features + ["Total_Cell_Count"],
    "ECD + total cell area": ecd_features + ["Total_Area_um2"],
    "ECD + all morphology": ecd_features + morphology_without_ecd,
}

print("MORPHOLOGY FEATURE EXPERIMENTS")
for name, features in morphology_feature_groups.items():
    missing = [
        feature for feature in features
        if feature not in X_train_reduced.columns
    ]

    print(f"\n{name}")
    print("Features:", features)
    print("Missing columns:", missing if missing else "None")

MORPHOLOGY FEATURE EXPERIMENTS

ECD only
Features: ['ECD_CD_cells_per_mm2']
Missing columns: None

ECD + CV
Features: ['ECD_CD_cells_per_mm2', 'CV']
Missing columns: None

ECD + HEX
Features: ['ECD_CD_cells_per_mm2', 'HEX_percent']
Missing columns: None

ECD + SD
Features: ['ECD_CD_cells_per_mm2', 'SD']
Missing columns: None

ECD + total cell count
Features: ['ECD_CD_cells_per_mm2', 'Total_Cell_Count']
Missing columns: None

ECD + total cell area
Features: ['ECD_CD_cells_per_mm2', 'Total_Area_um2']
Missing columns: None

ECD + all morphology
Features: ['ECD_CD_cells_per_mm2', 'CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2']
Missing columns: None


In [92]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

morphology_results = []

for group_name, features in morphology_feature_groups.items():
    X_group = X_train_reduced[features].copy()

    numeric_preprocessor = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_preprocessor, features)
        ],
        remainder="drop"
    )

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])

    scores = cross_validate(
        model,
        X_group,
        y_train,
        cv=rf_cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "macro_f1": "f1_macro"
        },
        n_jobs=-1,
        return_train_score=False
    )

    morphology_results.append({
        "Feature group": group_name,
        "Features": len(features),
        "Accuracy": scores["test_accuracy"].mean(),
        "Balanced accuracy": scores["test_balanced_accuracy"].mean(),
        "Macro F1": scores["test_macro_f1"].mean(),
        "Macro F1 SD": scores["test_macro_f1"].std()
    })

morphology_results_df = (
    pd.DataFrame(morphology_results)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

print("ECD + MORPHOLOGY — FIVE-FOLD CV RESULTS")
print(morphology_results_df.round(4).to_string(index=False))

ECD + MORPHOLOGY — FIVE-FOLD CV RESULTS
         Feature group  Features  Accuracy  Balanced accuracy  Macro F1  Macro F1 SD
  ECD + all morphology         6    0.8334             0.8199    0.8332       0.0300
             ECD + HEX         2    0.8104             0.8048    0.8139       0.0245
 ECD + total cell area         2    0.7935             0.7855    0.7944       0.0415
              ECD + SD         2    0.7839             0.7782    0.7909       0.0299
ECD + total cell count         2    0.7839             0.7793    0.7866       0.0354
              ECD only         1    0.7717             0.7722    0.7722       0.0270
              ECD + CV         2    0.7549             0.7648    0.7708       0.0276


In [93]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

hex_comparison_groups = {
    "ECD only": ["ECD_CD_cells_per_mm2"],
    "ECD + HEX": ["ECD_CD_cells_per_mm2", "HEX_percent"],
    "ECD + other morphology (no HEX)": [
        "ECD_CD_cells_per_mm2",
        "CV",
        "SD",
        "Total_Cell_Count",
        "Total_Area_um2",
    ],
}

hex_comparison_results = []

for group_name, features in hex_comparison_groups.items():
    X_group = X_train_reduced[features].copy()

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median"))
                ]),
                features,
            )
        ],
        remainder="drop",
    )

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )),
    ])

    scores = cross_validate(
        model,
        X_group,
        y_train,
        cv=rf_cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "macro_f1": "f1_macro",
        },
        n_jobs=-1,
        return_train_score=False,
    )

    hex_comparison_results.append({
        "Feature group": group_name,
        "Features": len(features),
        "Accuracy": scores["test_accuracy"].mean(),
        "Balanced accuracy": scores["test_balanced_accuracy"].mean(),
        "Macro F1": scores["test_macro_f1"].mean(),
        "Macro F1 SD": scores["test_macro_f1"].std(),
    })

hex_comparison_results_df = (
    pd.DataFrame(hex_comparison_results)
    .sort_values("Macro F1", ascending=False)
    .reset_index(drop=True)
)

print("HEX CONTRIBUTION — FIVE-FOLD CV")
print(hex_comparison_results_df.round(4).to_string(index=False))

HEX CONTRIBUTION — FIVE-FOLD CV
                  Feature group  Features  Accuracy  Balanced accuracy  Macro F1  Macro F1 SD
                      ECD + HEX         2    0.8104             0.8048    0.8139       0.0245
ECD + other morphology (no HEX)         5    0.8056             0.7952    0.8087       0.0335
                       ECD only         1    0.7717             0.7722    0.7722       0.0270


In [94]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline

per_class_groups = {
    "ECD only": ["ECD_CD_cells_per_mm2"],
    "ECD + HEX": ["ECD_CD_cells_per_mm2", "HEX_percent"],
    "ECD + all morphology": [
        "ECD_CD_cells_per_mm2",
        "CV",
        "HEX_percent",
        "SD",
        "Total_Cell_Count",
        "Total_Area_um2",
    ],
}

per_class_predictions = {}

for group_name, features in per_class_groups.items():
    X_group = X_train_reduced[features].copy()

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median"))
                ]),
                features,
            )
        ],
        remainder="drop",
    )

    model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        )),
    ])

    per_class_predictions[group_name] = cross_val_predict(
        model,
        X_group,
        y_train,
        cv=rf_cv,
        n_jobs=-1,
        method="predict",
    )

for group_name, predictions in per_class_predictions.items():
    print(f"\n{'=' * 65}")
    print(group_name.upper())
    print("=" * 65)

    print(classification_report(
        y_train,
        predictions,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4,
    ))

    print("Confusion matrix (actual rows, predicted columns):")
    print(confusion_matrix(
        y_train,
        predictions,
        labels=["A+", "A", "B"],
    ))


ECD ONLY
              precision    recall  f1-score   support

          A+     0.8431    0.7890    0.8152       436
           A     0.6401    0.7255    0.6801       255
           B     0.8397    0.8029    0.8209       137

    accuracy                         0.7717       828
   macro avg     0.7743    0.7725    0.7721       828
weighted avg     0.7800    0.7717    0.7745       828

Confusion matrix (actual rows, predicted columns):
[[344  81  11]
 [ 60 185  10]
 [  4  23 110]]

ECD + HEX
              precision    recall  f1-score   support

          A+     0.8657    0.8280    0.8464       436
           A     0.6780    0.7843    0.7273       255
           B     0.9483    0.8029    0.8696       137

    accuracy                         0.8104       828
   macro avg     0.8306    0.8051    0.8144       828
weighted avg     0.8216    0.8104    0.8136       828

Confusion matrix (actual rows, predicted columns):
[[361  73   2]
 [ 51 200   4]
 [  5  22 110]]

ECD + ALL MORPHOLOGY
 

In [95]:
numeric_comparison_features = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
]

print("MODEL COMPARISON SETUP")
print("Training rows:", len(y_train))
print("Feature count:", len(numeric_comparison_features))
print("Features:", numeric_comparison_features)
print("CV folds:", rf_cv.get_n_splits())
print("Random Forest random state:", 42)
print("Shared CV random state:", rf_cv.random_state)

print("\nMissing feature columns:")
print([
    feature for feature in numeric_comparison_features
    if feature not in X_train_reduced.columns
])

print("\nTarget counts:")
print(y_train.value_counts().reindex(["A+", "A", "B"]).to_string())

MODEL COMPARISON SETUP
Training rows: 828
Feature count: 6
Features: ['ECD_CD_cells_per_mm2', 'CV', 'HEX_percent', 'SD', 'Total_Cell_Count', 'Total_Area_um2']
CV folds: 5
Random Forest random state: 42
Shared CV random state: 42

Missing feature columns:
[]

Target counts:
Tissue_Grade_Clean
A+    436
A     255
B     137


In [96]:
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline

X_numeric_comparison = X_train_reduced[
    numeric_comparison_features
].copy()

# Preprocessing for both models is fitted within each CV fold.
numeric_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]),
            numeric_comparison_features
        )
    ],
    remainder="drop"
)

comparison_models = {
    "Logistic Regression": Pipeline(steps=[
        ("preprocessor", numeric_preprocessor),
        ("classifier", LogisticRegression(
            max_iter=5000,
            class_weight="balanced",
            random_state=42
        ))
    ]),
    "Random Forest": Pipeline(steps=[
        ("preprocessor", numeric_preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ))
    ])
}

algorithm_results = []
algorithm_fold_scores = {}

for model_name, model in comparison_models.items():
    scores = cross_validate(
        model,
        X_numeric_comparison,
        y_train,
        cv=rf_cv,
        scoring={
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "macro_f1": "f1_macro"
        },
        n_jobs=-1,
        return_train_score=False
    )

    algorithm_fold_scores[model_name] = scores["test_macro_f1"]

    algorithm_results.append({
        "Model": model_name,
        "Accuracy mean": scores["test_accuracy"].mean(),
        "Balanced accuracy mean": scores["test_balanced_accuracy"].mean(),
        "Macro F1 mean": scores["test_macro_f1"].mean(),
        "Macro F1 SD": scores["test_macro_f1"].std()
    })

algorithm_results_df = (
    pd.DataFrame(algorithm_results)
    .sort_values("Macro F1 mean", ascending=False)
    .reset_index(drop=True)
)

print("ALGORITHM COMPARISON — SIX NUMERIC FEATURES")
print(algorithm_results_df.round(4).to_string(index=False))

print("\nPER-FOLD MACRO F1")
print("Fold | Logistic Regression | Random Forest")
for i, (lr_score, rf_score) in enumerate(
    zip(
        algorithm_fold_scores["Logistic Regression"],
        algorithm_fold_scores["Random Forest"]
    ),
    start=1
):
    print(f"{i:>4} | {lr_score:.4f}              | {rf_score:.4f}")

ALGORITHM COMPARISON — SIX NUMERIC FEATURES
              Model  Accuracy mean  Balanced accuracy mean  Macro F1 mean  Macro F1 SD
      Random Forest         0.8334                  0.8199         0.8332       0.0300
Logistic Regression         0.7017                  0.7204         0.7085       0.0293

PER-FOLD MACRO F1
Fold | Logistic Regression | Random Forest
   1 | 0.7285              | 0.8554
   2 | 0.7308              | 0.8386
   3 | 0.6777              | 0.7744
   4 | 0.7373              | 0.8531
   5 | 0.6681              | 0.8442


In [97]:
from sklearn.base import clone
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix

oof_predictions = {}

for model_name, model in comparison_models.items():
    oof_predictions[model_name] = cross_val_predict(
        model,
        X_numeric_comparison,
        y_train,
        cv=rf_cv,
        method="predict",
        n_jobs=-1
    )

for model_name, predictions in oof_predictions.items():
    print("\n" + "=" * 65)
    print(model_name.upper())
    print("=" * 65)

    print("\nPer-class classification report:")
    print(classification_report(
        y_train,
        predictions,
        labels=["A+", "A", "B"],
        zero_division=0,
        digits=4
    ))

    print("Confusion matrix (actual rows, predicted columns):")
    print(confusion_matrix(
        y_train,
        predictions,
        labels=["A+", "A", "B"]
    ))


LOGISTIC REGRESSION

Per-class classification report:
              precision    recall  f1-score   support

          A+     0.8182    0.7018    0.7556       436
           A     0.5294    0.6353    0.5775       255
           B     0.7635    0.8248    0.7930       137

    accuracy                         0.7017       828
   macro avg     0.7037    0.7206    0.7087       828
weighted avg     0.7202    0.7017    0.7069       828

Confusion matrix (actual rows, predicted columns):
[[306 128   2]
 [ 60 162  33]
 [  8  16 113]]

RANDOM FOREST

Per-class classification report:
              precision    recall  f1-score   support

          A+     0.8733    0.8693    0.8713       436
           A     0.7153    0.7882    0.7500       255
           B     0.9735    0.8029    0.8800       137

    accuracy                         0.8333       828
   macro avg     0.8540    0.8201    0.8338       828
weighted avg     0.8412    0.8333    0.8354       828

Confusion matrix (actual rows, predic

In [98]:
import pandas as pd

# Use the existing out-of-fold Random Forest predictions.
grade_a_error_mask = (
    (y_train.to_numpy() == "A") &
    (oof_predictions["Random Forest"] != "A")
)

grade_a_errors = X_train_reduced.loc[
    y_train.index[grade_a_error_mask]
].copy()

grade_a_errors["Actual_Grade"] = "A"
grade_a_errors["Predicted_Grade"] = (
    oof_predictions["Random Forest"][grade_a_error_mask]
)

print("GRADE A ERROR ANALYSIS")
print("Actual Grade A cases:", int((y_train == "A").sum()))
print("Misclassified Grade A cases:", len(grade_a_errors))
print(
    "Grade A error rate:",
    f"{len(grade_a_errors) / (y_train == 'A').sum():.2%}"
)

print("\nPredicted grades for misclassified Grade A cases:")
print(
    grade_a_errors["Predicted_Grade"]
    .value_counts()
    .reindex(["A+", "B"], fill_value=0)
    .to_string()
)

print("\nECD and morphology summary — misclassified Grade A cases:")
features_to_inspect = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
]

print(grade_a_errors[features_to_inspect].describe().round(2).to_string())

GRADE A ERROR ANALYSIS
Actual Grade A cases: 255
Misclassified Grade A cases: 54
Grade A error rate: 21.18%

Predicted grades for misclassified Grade A cases:
Predicted_Grade
A+    51
B      3

ECD and morphology summary — misclassified Grade A cases:
       ECD_CD_cells_per_mm2     CV  HEX_percent      SD  Total_Cell_Count  Total_Area_um2
count                 54.00  54.00        54.00   54.00             54.00           54.00
mean                2809.04  38.50        53.50  138.11            105.65        37831.15
std                  311.78   3.98         5.85   18.17             18.38         6979.88
min                 2304.00  30.00        42.00  105.00             25.00         9177.00
25%                 2574.25  36.00        50.00  123.00            104.00        34763.50
50%                 2732.00  39.00        53.00  135.00            106.50        38826.00
75%                 2930.75  40.75        57.00  148.75            115.00        41935.25
max                 3774.00 

In [99]:
# Compare correctly and incorrectly classified Grade A cases.
grade_a_correct_mask = (
    (y_train.to_numpy() == "A") &
    (oof_predictions["Random Forest"] == "A")
)

grade_a_correct = X_train_reduced.loc[
    y_train.index[grade_a_correct_mask]
].copy()

features_to_compare = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2",
]

comparison_summary = pd.DataFrame({
    "Correctly classified A": grade_a_correct[features_to_compare].median(),
    "Misclassified A": grade_a_errors[features_to_compare].median(),
})

comparison_summary["Difference"] = (
    comparison_summary["Misclassified A"]
    - comparison_summary["Correctly classified A"]
)

print("GRADE A — CORRECT VS MISCLASSIFIED")
print("Correctly classified cases:", len(grade_a_correct))
print("Misclassified cases:", len(grade_a_errors))

print("\nMedian feature values:")
print(comparison_summary.round(2).to_string())

print("\nECD quartiles by group:")
print(
    pd.DataFrame({
        "Correctly classified A": grade_a_correct[
            "ECD_CD_cells_per_mm2"
        ].quantile([0.25, 0.50, 0.75]),
        "Misclassified A": grade_a_errors[
            "ECD_CD_cells_per_mm2"
        ].quantile([0.25, 0.50, 0.75]),
    }).round(2).to_string()
)

GRADE A — CORRECT VS MISCLASSIFIED
Correctly classified cases: 201
Misclassified cases: 54

Median feature values:
                      Correctly classified A  Misclassified A  Difference
ECD_CD_cells_per_mm2                  2358.0           2732.0       374.0
CV                                      38.0             39.0         1.0
HEX_percent                             53.0             53.0         0.0
SD                                     158.0            135.0       -23.0
Total_Cell_Count                       108.0            106.5        -1.5
Total_Area_um2                       45958.5          38826.0     -7132.5

ECD quartiles by group:
      Correctly classified A  Misclassified A
0.25                  2288.0          2574.25
0.50                  2358.0          2732.00
0.75                  2451.0          2930.75


In [100]:
# Compare ECD ranges in correctly and incorrectly classified Grade A cases.

def summarize_grade_a_ecd(data, group_name):
    ecd = data["ECD_CD_cells_per_mm2"]

    print(f"\n{group_name}")
    print("Cases:", len(data))
    print("ECD > 2500:", int((ecd > 2500).sum()))
    print("ECD <= 2500:", int((ecd <= 2500).sum()))
    print("ECD > 3000:", int((ecd > 3000).sum()))
    print("ECD missing:", int(ecd.isna().sum()))
    print("Percentage > 2500:", f"{(ecd > 2500).mean():.2%}")

print("ECD DISTRIBUTION — GRADE A")
summarize_grade_a_ecd(grade_a_correct, "Correctly classified as A")
summarize_grade_a_ecd(grade_a_errors, "Misclassified as another grade")

ECD DISTRIBUTION — GRADE A

Correctly classified as A
Cases: 201
ECD > 2500: 25
ECD <= 2500: 143
ECD > 3000: 11
ECD missing: 33
Percentage > 2500: 12.44%

Misclassified as another grade
Cases: 54
ECD > 2500: 50
ECD <= 2500: 4
ECD > 3000: 11
ECD missing: 0
Percentage > 2500: 92.59%


In [101]:

# Separate Grade A errors by the grade predicted by the Random Forest

grade_a_error_details = pd.DataFrame({
    "Actual": y_train.to_numpy(),
    "Predicted": oof_predictions["Random Forest"]
}, index=y_train.index)

grade_a_error_details = grade_a_error_details[
    (grade_a_error_details["Actual"] == "A") &
    (grade_a_error_details["Predicted"] != "A")
].copy()

grade_a_error_details["ECD"] = labelled_df.loc[
    grade_a_error_details.index, "ECD_CD_cells_per_mm2"
]

print("GRADE A ERRORS BY PREDICTED GRADE")

for predicted_grade, group in grade_a_error_details.groupby("Predicted"):
    ecd = group["ECD"]

    print(f"\nPredicted as {predicted_grade}")
    print("Cases:", len(group))
    print("ECD missing:", int(ecd.isna().sum()))
    print("ECD median:", ecd.median())
    print("ECD minimum:", ecd.min())
    print("ECD maximum:", ecd.max())
    print("ECD > 2500:", int((ecd > 2500).sum()))
    print("ECD > 3000:", int((ecd > 3000).sum()))


GRADE A ERRORS BY PREDICTED GRADE

Predicted as A+
Cases: 51
ECD missing: 0
ECD median: 2755.0
ECD minimum: 2500.0
ECD maximum: 3774.0
ECD > 2500: 49
ECD > 3000: 11

Predicted as B
Cases: 3
ECD missing: 0
ECD median: 2364.0
ECD minimum: 2304.0
ECD maximum: 2725.0
ECD > 2500: 1
ECD > 3000: 0


In [102]:

# Compare the three Grade A -> B errors with correctly classified Grade A cases

numeric_features = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2"
]

grade_a_correct_idx = y_train.index[
    (y_train.to_numpy() == "A") &
    (oof_predictions["Random Forest"] == "A")
]

grade_a_to_b_idx = grade_a_error_details.index[
    grade_a_error_details["Predicted"] == "B"
]

comparison = pd.DataFrame({
    "Correctly_predicted_A": labelled_df.loc[
        grade_a_correct_idx, numeric_features
    ].median(),
    "Grade_A_predicted_B": labelled_df.loc[
        grade_a_to_b_idx, numeric_features
    ].median()
})

comparison["Difference"] = (
    comparison["Grade_A_predicted_B"] -
    comparison["Correctly_predicted_A"]
)

print("MORPHOLOGY COMPARISON: CORRECT GRADE A vs GRADE A PREDICTED AS B")
print(comparison.round(2).to_string())


MORPHOLOGY COMPARISON: CORRECT GRADE A vs GRADE A PREDICTED AS B
                      Correctly_predicted_A  Grade_A_predicted_B  Difference
ECD_CD_cells_per_mm2                 2358.0               2364.0         6.0
CV                                     38.0                 38.0         0.0
HEX_percent                            53.0                 44.0        -9.0
SD                                    158.0                167.0         9.0
Total_Cell_Count                      108.0                 57.0       -51.0
Total_Area_um2                      45958.5              24740.0    -21218.5


In [103]:

# Inspect the individual Grade A cases predicted as B

error_indices = grade_a_error_details.index[
    grade_a_error_details["Predicted"] == "B"
]

inspection_columns = [
    "Tissue_ID",
    "Image_ID",
    "Tissue_Grade_Clean",
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2"
]

available_columns = [
    col for col in inspection_columns
    if col in labelled_df.columns
]

print("INDIVIDUAL GRADE A -> B ERRORS")
print(
    labelled_df.loc[error_indices, available_columns]
    .to_string(index=False)
)


INDIVIDUAL GRADE A -> B ERRORS
          Tissue_ID               Image_ID Tissue_Grade_Clean  ECD_CD_cells_per_mm2   CV  HEX_percent    SD  Total_Cell_Count  Total_Area_um2
NEB-2023-06-15151 R NEB-2023-06-15151 R_SP                  A                2364.0 43.0         42.0 183.0              66.0         27917.0
NEB-2023-08-15308 R NEB-2023-08-15308 R_SP                  A                2304.0 38.0         45.0 167.0              57.0         24740.0
NEB-2023-06-15144 R NEB-2023-06-15144 R_SP                  A                2725.0 34.0         44.0 123.0              25.0          9177.0


In [104]:

# Check how unusual the three cases are across the labelled dataset

check_features = [
    "ECD_CD_cells_per_mm2",
    "CV",
    "HEX_percent",
    "SD",
    "Total_Cell_Count",
    "Total_Area_um2"
]

case_ids = [
    "NEB-2023-06-15151 R",
    "NEB-2023-08-15308 R",
    "NEB-2023-06-15144 R"
]

for tissue_id in case_ids:
    row = labelled_df[labelled_df["Tissue_ID"] == tissue_id]

    print(f"\nTissue ID: {tissue_id}")

    if row.empty:
        print("Tissue ID not found.")
        continue

    for feature in check_features:
        values = pd.to_numeric(labelled_df[feature], errors="coerce")
        value = pd.to_numeric(row[feature], errors="coerce").iloc[0]

        percentile = (values.dropna() <= value).mean() * 100

        print(
            f"{feature}: {value} "
            f"| Dataset percentile: {percentile:.1f}"
        )



Tissue ID: NEB-2023-06-15151 R
ECD_CD_cells_per_mm2: 2364.0 | Dataset percentile: 27.4
CV: 43.0 | Dataset percentile: 87.9
HEX_percent: 42.0 | Dataset percentile: 3.6
SD: 183.0 | Dataset percentile: 87.6
Total_Cell_Count: 66.0 | Dataset percentile: 2.7
Total_Area_um2: 27917.0 | Dataset percentile: 3.9

Tissue ID: NEB-2023-08-15308 R
ECD_CD_cells_per_mm2: 2304.0 | Dataset percentile: 21.9
CV: 38.0 | Dataset percentile: 57.2
HEX_percent: 45.0 | Dataset percentile: 9.4
SD: 167.0 | Dataset percentile: 77.3
Total_Cell_Count: 57.0 | Dataset percentile: 2.1
Total_Area_um2: 24740.0 | Dataset percentile: 2.3

Tissue ID: NEB-2023-06-15144 R
ECD_CD_cells_per_mm2: 2725.0 | Dataset percentile: 64.3
CV: 34.0 | Dataset percentile: 21.5
HEX_percent: 44.0 | Dataset percentile: 6.3
SD: 123.0 | Dataset percentile: 18.3
Total_Cell_Count: 25.0 | Dataset percentile: 0.9
Total_Area_um2: 9177.0 | Dataset percentile: 0.5
